# 2026 호르무즈 해협 영어 해외언론 텍스트마이닝

**연구 기간:** 2026-01-01 ~ 2026-08-31  
**핵심 키워드:** `Strait of Hormuz`, `Hormuz Strait`  
**원본 데이터:** 기존에 수집한 `01_newscatcher_raw.csv`

## 이 노트북의 원칙

이 버전은 기존 노트북의 **NewsCatcher 재수집 및 Diffbot 전체 수집을 제거**하고, 이미 확보한 원본 CSV에서 연구용 데이터셋을 다시 구성한다.

핵심 변경점:

1. NewsCatcher API를 다시 호출하지 않는다.
2. URL 중복만 먼저 정리하고, 원본 NewsCatcher 메타데이터를 최대한 보존한다.
3. `description`과 NewsCatcher의 잘린 `content`를 서로 다른 컬럼으로 보존한다.
4. 동일 제목을 발견해도 기사를 삭제하지 않고 **중복 그룹으로 표시**한다.
5. `hormuz_relevance_score`는 삭제 기준이 아니라 분석용 메타데이터/추출 우선순위로 사용한다.
6. 본문 추출은 이 노트북에서 42만 건 전체를 바로 실행하지 않는다.
7. 먼저 층화 표본으로 **Fundus / news-please / trafilatura** 등의 추출 성능을 비교한 뒤 기본 추출기를 결정한다.
8. 본문 확보 후 언어, 본문 품질, 신디케이션/중복 여부를 판단한다.

> `01_newscatcher_raw.csv`는 원본이므로 이 노트북에서 수정하거나 덮어쓰지 않는다.


## 1. 패키지와 프로젝트 경로

이 노트북은 기존 NewsCatcher 결과를 읽는 것이 목적이므로 API 수집 패키지는 사용하지 않는다.

처음 실행할 때 필요한 최소 패키지만 설치한다.


In [ ]:
%pip install -q pandas numpy requests python-dotenv tqdm langdetect scikit-learn

## 2. 기본 설정 및 데이터 위치 확인

노트북의 현재 작업 폴더가 `Hormuz_Research`이면 자동으로 찾는다.  
VS Code에서 다른 폴더를 작업 디렉터리로 잡은 경우에는 상위 폴더를 탐색한다.

**중요:** 아래 셀에서 찾는 파일은 기존 `01_newscatcher_raw.csv` 하나다.


In [1]:
from pathlib import Path
import hashlib
import re
from urllib.parse import urlsplit, urlunsplit

import numpy as np
import pandas as pd

PROJECT_NAME = "Hormuz_Research"
RAW_FILENAME = "01_newscatcher_raw.csv"

START_DATE = "2026-01-01"
END_DATE = "2026-08-31"

# 현재 위치부터 상위 폴더까지 탐색
candidate_roots = [Path.cwd()] + list(Path.cwd().parents)

PROJECT_ROOT = None
for root in candidate_roots:
    if root.name == PROJECT_NAME and (root / "hormuz_2026_data" / RAW_FILENAME).exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    # 현재 폴더 자체가 프로젝트 폴더인 경우
    if (Path.cwd() / "hormuz_2026_data" / RAW_FILENAME).exists():
        PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Hormuz_Research/hormuz_2026_data/01_newscatcher_raw.csv를 찾지 못했습니다.\n"
        "VS Code에서 Hormuz_Research 폴더를 열고 다시 실행하세요."
    )

DATA_DIR = PROJECT_ROOT / "hormuz_2026_data"
RAW_NEWS_FILE = DATA_DIR / RAW_FILENAME

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RAW_NEWS_FILE:", RAW_NEWS_FILE)


PROJECT_ROOT: c:\Users\황태하\Hormuz_Research
DATA_DIR: c:\Users\황태하\Hormuz_Research\hormuz_2026_data
RAW_NEWS_FILE: c:\Users\황태하\Hormuz_Research\hormuz_2026_data\01_newscatcher_raw.csv


## 3. 원본 NewsCatcher CSV 로드

이 단계에서는 원본 레코드를 삭제하지 않는다.

원본에 존재하는 주요 메타데이터를 `nc_` 접두사로 명확하게 보존한다. 특히 기존 노트북에서 버려졌던 다음 항목을 유지한다.

- `word_count`
- `paid_content`
- `country`
- `is_opinion`
- `rank`
- `duplicate_count`
- NewsCatcher 원본 `id`
- `published_date_precision`
- `description`
- `content`
- `is_content_truncated`

또한 `content`는 기사 본문으로 취급하지 않는다. NewsCatcher가 반환한 짧은 콘텐츠/스니펫으로만 보존한다.


In [2]:
raw_df = pd.read_csv(
    RAW_NEWS_FILE,
    encoding="utf-8-sig",
    low_memory=False,
)

print("=" * 70)
print("RAW DATA")
print("=" * 70)
print(f"원본 레코드 수: {len(raw_df):,}")
print(f"컬럼 수: {len(raw_df.columns)}")
print()
print(raw_df.columns.tolist())

required_raw = ["title", "link", "published_date", "language", "description", "content"]
missing_raw = [c for c in required_raw if c not in raw_df.columns]

if missing_raw:
    raise KeyError(f"원본 CSV에 필요한 컬럼이 없습니다: {missing_raw}")

print()
print("필수 원본 컬럼 확인 완료")


RAW DATA
원본 레코드 수: 440,260
컬럼 수: 32

['title', 'author', 'authors', 'journalists', 'published_date', 'published_date_precision', 'updated_date', 'updated_date_precision', 'link', 'domain_url', 'full_domain_url', 'name_source', 'is_headline', 'paid_content', 'parent_url', 'country', 'rights', 'rank', 'media', 'language', 'description', 'content', 'word_count', 'is_opinion', 'twitter_account', 'all_links', 'all_domain_links', 'id', 'score', 'duplicate_count', 'duplicate_articles_group_id', 'is_content_truncated']

필수 원본 컬럼 확인 완료


## 4. URL 정규화

URL 정규화는 **같은 URL의 기술적 변형을 비교하기 위한 단계**다.

이번 버전에서는 원래 URL의 query string을 임의로 제거하지 않는다.  
따라서 URL 정규화만으로 과도하게 서로 다른 URL을 합치지 않는다.

- scheme 소문자화
- hostname 소문자화
- fragment 제거
- 마지막 `/` 제거

URL 중복은 이후 하나의 대표 행으로 만들되, 원본 NewsCatcher ID와 중복 개수를 별도로 보존한다.


In [3]:
def normalize_url(url):
    if pd.isna(url):
        return None

    url = str(url).strip()
    if not url:
        return None

    try:
        parsed = urlsplit(url)

        if parsed.scheme.lower() not in {"http", "https"}:
            return None

        if not parsed.netloc:
            return None

        return urlunsplit((
            parsed.scheme.lower(),
            parsed.netloc.lower(),
            parsed.path.rstrip("/") or "/",
            parsed.query,
            "",
        ))
    except Exception:
        return None


raw_df["url_raw"] = raw_df["link"].astype("string")
raw_df["url"] = raw_df["url_raw"].apply(normalize_url)

print("유효 URL:", raw_df["url"].notna().sum())
print("유효 URL 없음:", raw_df["url"].isna().sum())


유효 URL: 440260
유효 URL 없음: 0


## 5. URL 기준 기사 인덱스 재구성

여기서 **URL 중복만 제거**한다.

같은 URL이 여러 NewsCatcher 레코드에 존재할 경우:

- 원본 NewsCatcher ID들을 `nc_ids`에 보존
- 원본 레코드 수를 `nc_duplicate_count`에 기록
- 서로 다른 제목이 있었는지 `nc_multiple_titles`로 기록
- 대표 행은 메타데이터가 더 많이 채워진 행을 우선하고, 동률이면 최초 수집/발행 정보를 고려해 결정

이 단계는 제목 중복 제거와 다르다. 동일 제목을 가진 서로 다른 URL은 삭제하지 않는다.


In [4]:
# 원본 필드 → 연구용 필드
RENAME_MAP = {
    "id": "nc_id",
    "title": "title",
    "author": "nc_author",
    "authors": "nc_authors",
    "journalists": "nc_journalists",
    "published_date": "published_date",
    "published_date_precision": "nc_published_date_precision",
    "updated_date": "nc_updated_date",
    "updated_date_precision": "nc_updated_date_precision",
    "description": "nc_description",
    "content": "nc_content_snippet",
    "word_count": "nc_word_count",
    "is_opinion": "nc_is_opinion",
    "paid_content": "nc_paid_content",
    "country": "nc_country",
    "rights": "nc_rights",
    "rank": "nc_rank",
    "media": "nc_media",
    "language": "nc_language",
    "name_source": "nc_source_name",
    "domain_url": "nc_domain_url",
    "full_domain_url": "nc_full_domain_url",
    "parent_url": "nc_parent_url",
    "is_headline": "nc_is_headline",
    "all_links": "nc_all_links",
    "all_domain_links": "nc_all_domain_links",
    "score": "nc_score",
    "duplicate_count": "nc_duplicate_count_vendor",
    "duplicate_articles_group_id": "nc_duplicate_group_id_vendor",
    "is_content_truncated": "nc_is_content_truncated",
    "link": "nc_original_link",
}

work = raw_df.copy()

for raw_col, new_col in RENAME_MAP.items():
    if raw_col in work.columns:
        work[new_col] = work[raw_col]

# 메타데이터 완성도: 대표 URL 행을 결정할 때만 사용
metadata_priority_cols = [
    "title", "nc_description", "nc_content_snippet", "published_date",
    "nc_language", "nc_source_name", "nc_country", "nc_word_count",
]

work["_metadata_completeness"] = work[metadata_priority_cols].notna().sum(axis=1)

# 날짜는 정렬용으로만 사용. 최종 연구 날짜의 기준은 NewsCatcher published_date.
work["_published_sort"] = pd.to_datetime(
    work["published_date"], errors="coerce", utc=True
)

# 대표 행을 결정하기 위한 안정적인 정렬
work = work.sort_values(
    by=["url", "_metadata_completeness", "_published_sort"],
    ascending=[True, False, True],
    kind="mergesort",
)

# URL별 원본 ID/제목/도메인 정보 보존
url_groups = work.groupby("url", dropna=True)

url_meta = url_groups.agg(
    nc_ids=("nc_id", lambda s: " | ".join(pd.Series(s.dropna().astype(str)).drop_duplicates())),
    nc_url_record_count=("url", "size"),
    nc_multiple_titles=("title", lambda s: s.dropna().astype(str).nunique() > 1),
    nc_title_count=("title", lambda s: s.dropna().astype(str).nunique()),
    nc_source_count=("nc_source_name", lambda s: s.dropna().astype(str).nunique()),
).reset_index()

# URL별 대표 행 1개
article_index = (
    work.drop_duplicates(subset=["url"], keep="first")
        .drop(columns=["_metadata_completeness", "_published_sort"], errors="ignore")
        .merge(url_meta, on="url", how="left")
        .reset_index(drop=True)
)

# 연구용 article_id는 정규화 URL 기반으로 생성
article_index["article_id"] = article_index["url"].apply(
    lambda u: hashlib.sha1(str(u).encode("utf-8")).hexdigest()[:16]
)

print("=" * 70)
print("URL DEDUPLICATION")
print("=" * 70)
print(f"원본 레코드: {len(raw_df):,}")
print(f"유효 URL 레코드: {raw_df['url'].notna().sum():,}")
print(f"URL 중복 제거 후 기사: {len(article_index):,}")
print(f"URL 중복으로 합쳐진 레코드: {raw_df['url'].notna().sum() - len(article_index):,}")
print(f"동일 URL에서 제목이 2개 이상인 그룹: {article_index['nc_multiple_titles'].sum():,}")


URL DEDUPLICATION
원본 레코드: 440,260
유효 URL 레코드: 440,260
URL 중복 제거 후 기사: 427,875
URL 중복으로 합쳐진 레코드: 12,385
동일 URL에서 제목이 2개 이상인 그룹: 1,147


## 6. 기본 메타데이터와 연구 기간 플래그

이 단계에서는 기존처럼 제목 길이, 영어 여부 등을 이유로 기사를 대량 삭제하지 않는다.

특히 다음은 **플래그**로만 관리한다.

- 제목이 지나치게 짧음
- 제목이 지나치게 김
- URL이 동영상/라이브/태그 페이지처럼 보임
- 알려진 aggregator/republication 도메인
- NewsCatcher 관련성 점수 0/1/2

연구 기간 밖의 기사와 URL이 없는 원본 레코드는 연구 대상 후보에서 제외할 수 있지만, 원본 CSV에는 영향을 주지 않는다.


In [5]:
def get_domain(url):
    try:
        return urlsplit(str(url)).netloc.lower().split(":")[0]
    except Exception:
        return ""


def normalize_title(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def hormuz_relevance(title, description, snippet):
    text = " ".join([
        str(title) if pd.notna(title) else "",
        str(description) if pd.notna(description) else "",
        str(snippet) if pd.notna(snippet) else "",
    ]).lower()

    if "strait of hormuz" in text or "hormuz strait" in text:
        return 2
    if "hormuz" in text:
        return 1
    return 0


article_index["domain"] = article_index["url"].apply(get_domain)

article_index["normalized_title"] = article_index["title"].apply(normalize_title)

article_index["title_char_count"] = (
    article_index["title"].fillna("").astype(str).str.len()
)

article_index["flag_title_short"] = article_index["title_char_count"] < 10
article_index["flag_title_very_long"] = article_index["title_char_count"] > 300

article_index["flag_url_video"] = article_index["url"].str.lower().str.contains(
    r"/video[s]?/|/watch/|/live/", regex=True, na=False
)

article_index["flag_url_non_article"] = article_index["url"].str.lower().str.contains(
    r"/tag/|/tags/|/category/|/author/|/search/", regex=True, na=False
)

AGGREGATOR_REVIEW_DOMAINS = {
    "wn.com",
    "headtopics.com",
    "newsbreak.com",
    "msn.com",
    "yahoo.com",
    "finance.yahoo.com",
    "investing.com",
    "tradingview.com",
    "dailyhunt.in",
}

article_index["flag_aggregator_review"] = article_index["domain"].apply(
    lambda d: (
        d in AGGREGATOR_REVIEW_DOMAINS
        or any(d.endswith("." + x) for x in AGGREGATOR_REVIEW_DOMAINS)
    )
)

article_index["hormuz_relevance_score"] = [
    hormuz_relevance(t, d, s)
    for t, d, s in zip(
        article_index["title"],
        article_index["nc_description"],
        article_index["nc_content_snippet"],
    )
]

published_dt = pd.to_datetime(
    article_index["published_date"],
    errors="coerce",
    utc=True,
)

start_dt = pd.Timestamp(START_DATE, tz="UTC")
end_dt_exclusive = pd.Timestamp(END_DATE, tz="UTC") + pd.Timedelta(days=1)

article_index["in_research_period"] = (
    published_dt >= start_dt
) & (
    published_dt < end_dt_exclusive
)

article_index["flag_missing_title"] = (
    article_index["title"].fillna("").astype(str).str.strip().eq("")
)

article_index["flag_missing_description"] = article_index["nc_description"].isna()
article_index["flag_paid_content"] = (
    article_index["nc_paid_content"].fillna(False).astype(str).str.lower().isin(
        {"true", "1", "yes"}
    )
)

print("기사 수:", len(article_index))
print("연구 기간 내 기사:", article_index["in_research_period"].sum())
print("연구 기간 밖/날짜 미상:", (~article_index["in_research_period"]).sum())


기사 수: 427875
연구 기간 내 기사: 427875
연구 기간 밖/날짜 미상: 0


## 7. 동일 제목은 삭제하지 않고 그룹화

기존 노트북은 `normalized_title.duplicated(keep="first")` 방식으로 제목이 같은 기사를 삭제했다.

이 버전에서는 그렇게 하지 않는다.

대신 다음 정보를 만든다.

- `title_dup_group_id`
- `title_dup_group_size`
- `title_dup_n_domains`
- `title_dup_flag`

따라서 서로 다른 언론사의 같은 제목, 같은 기사의 재게시, 실제 중복 가능성을 나중에 본문 확보 후 판단할 수 있다.


In [6]:
valid_title_mask = article_index["normalized_title"].ne("")

article_index["title_dup_group_id"] = pd.NA
article_index.loc[valid_title_mask, "title_dup_group_id"] = (
    article_index.loc[valid_title_mask, "normalized_title"]
    .apply(lambda x: hashlib.sha1(x.encode("utf-8")).hexdigest()[:16])
)

title_group_stats = (
    article_index.loc[valid_title_mask]
    .groupby("title_dup_group_id")
    .agg(
        title_dup_group_size=("article_id", "size"),
        title_dup_n_domains=("domain", "nunique"),
    )
    .reset_index()
)

article_index = article_index.merge(
    title_group_stats,
    on="title_dup_group_id",
    how="left",
)

article_index["title_dup_group_size"] = (
    article_index["title_dup_group_size"].fillna(1).astype(int)
)

article_index["title_dup_n_domains"] = (
    article_index["title_dup_n_domains"].fillna(1).astype(int)
)

article_index["title_dup_flag"] = article_index["title_dup_group_size"] > 1

print("=" * 70)
print("TITLE DUPLICATE GROUPING")
print("=" * 70)
print("제목 중복 그룹에 포함된 기사:", article_index["title_dup_flag"].sum())
print("중복 제목 그룹 수:", title_group_stats["title_dup_group_id"].nunique())

print()
display(
    article_index.loc[
        article_index["title_dup_flag"],
        [
            "article_id", "title", "domain",
            "title_dup_group_size", "title_dup_n_domains"
        ]
    ].head(20)
)


TITLE DUPLICATE GROUPING
제목 중복 그룹에 포함된 기사: 76082
중복 제목 그룹 수: 380852



,article_id,title,domain,title_dup_group_size,title_dup_n_domains
4,920b4bcd9ed129af,UNDP urges countries to broaden fiscal space &...,absafricatv.com,2,2
28,d821f0238b3f7826,UK Trade Secretary Peter Kyle to visit India o...,alabama.statenews.net,14,14
29,57f6fdd4ca749cc9,US strikes Iran for second night as Trump says...,alabama.statenews.net,3,3
30,08356ed0fc23a447,Iran Targets Base Housing US Forces In Jordan,alabama.statenews.net,2,2
31,f487956d8fda030e,Strait of Hormuz crisis: What is the current s...,alabama.statenews.net,14,14
32,a697fce873b17cfe,US CPI inflation cools to 3.5% in June as ener...,alabama.statenews.net,41,41
33,1ce446a33a0fbbdd,Not fair that we would be guarding strait with...,alabama.statenews.net,22,22
34,39d466eba5169cf6,"Trump says US ready to protect Iraq ""if need b...",alabama.statenews.net,35,35
35,9e3fd272de244c11,Union Minister Sarbananda Sonowal orders 'Seaf...,alabama.statenews.net,20,20
36,b5b265edb623b2ad,"""Strait of Hormuz could reopen by tomorrow, de...",alabama.statenews.net,13,13


## 8. 영어/비영어 판정은 본문 확보 후 수행

NewsCatcher의 `language=en`은 유용한 메타데이터지만 기사 본문 전체가 영어라는 것을 확정하는 기준으로 사용하지 않는다.

따라서 여기서는 원본 언어값을 그대로 보존하고, 본문 추출 후 `language_detected`를 별도로 계산한다.

아래 `langdetect` 테스트 함수는 최종 삭제가 아니라 **본문 품질 진단용**이다.


In [7]:
from langdetect import detect, LangDetectException


def detect_language_safe(text):
    text = str(text or "").strip()

    if len(text) < 50:
        return None

    try:
        return detect(text)
    except LangDetectException:
        return None
    except Exception:
        return None


print("NewsCatcher language 분포:")
display(
    article_index["nc_language"]
    .fillna("(missing)")
    .astype(str)
    .str.lower()
    .value_counts()
    .head(20)
)


NewsCatcher language 분포:


nc_language
en    427875
Name: count, dtype: int64

## 9. 메타데이터 진단

이 셀은 기사를 삭제하지 않는다.

확인할 항목:

- 도메인 집중도
- 국가 분포
- 월별 분포
- NewsCatcher 관련성 점수
- `paid_content`
- NewsCatcher `word_count`
- 제목 중복 그룹
- aggregator/republication 검토 플래그

특정 도메인의 기사 수가 많다는 사실만으로 편향이라고 판단하지 않는다. 이후 본문 추출 성공률과 실제 기사 유형을 함께 확인한다.


In [8]:
diagnostic = article_index[
    article_index["in_research_period"]
].copy()

diagnostic["published_dt"] = pd.to_datetime(
    diagnostic["published_date"], errors="coerce", utc=True
)

diagnostic["month"] = diagnostic["published_dt"].dt.to_period("M").astype(str)

print("=" * 70)
print("METADATA DIAGNOSTIC")
print("=" * 70)

print(f"연구 기간 내 URL 중복 제거 기사: {len(diagnostic):,}")
print(f"유일 도메인 수: {diagnostic['domain'].nunique():,}")

print()
print("상위 30개 도메인:")
display(
    diagnostic["domain"]
    .value_counts()
    .head(30)
    .rename_axis("domain")
    .reset_index(name="article_count")
)

print()
print("상위 도메인 집중도:")
domain_counts = diagnostic["domain"].value_counts()
for n in [10, 20, 50, 100]:
    top_n = domain_counts.head(n).sum()
    ratio = top_n / len(diagnostic) * 100
    print(f"상위 {n:>3}개: {top_n:>8,}건 ({ratio:6.2f}%)")

print()
print("호르무즈 관련성 점수:")
display(
    diagnostic["hormuz_relevance_score"]
    .value_counts()
    .sort_index()
    .rename_axis("score")
    .reset_index(name="article_count")
)

print()
print("월별 기사 수:")
display(
    diagnostic["month"]
    .value_counts()
    .sort_index()
    .rename_axis("month")
    .reset_index(name="article_count")
)

print()
print("국가 분포 상위:")
display(
    diagnostic["nc_country"]
    .fillna("(unknown)")
    .astype(str)
    .value_counts()
    .head(20)
    .rename_axis("country")
    .reset_index(name="article_count")
)

print()
print("NewsCatcher word_count 요약:")
display(
    pd.to_numeric(diagnostic["nc_word_count"], errors="coerce")
    .describe()
    .to_frame("nc_word_count")
)

print()
print("paid_content:")
display(
    diagnostic["flag_paid_content"]
    .value_counts()
    .rename_axis("paid_content")
    .reset_index(name="article_count")
)


METADATA DIAGNOSTIC
연구 기간 내 URL 중복 제거 기사: 427,875
유일 도메인 수: 17,602

상위 30개 도메인:


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11164\74421316.py:9: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  diagnostic["month"] = diagnostic["published_dt"].dt.to_period("M").astype(str)


,domain,article_count
0,article.wn.com,16835
1,news-pravda.com,8460
2,www.msn.com,5642
3,www.vietnam.vn,3494
4,www.yahoo.com,3282
5,finance.yahoo.com,3173
6,timesofindia.indiatimes.com,3133
7,custommapposter.com,2847
8,www.newsbreak.com,2781
9,economictimes.indiatimes.com,2569



상위 도메인 집중도:
상위  10개:   52,216건 ( 12.20%)
상위  20개:   69,553건 ( 16.26%)
상위  50개:   99,871건 ( 23.34%)
상위 100개:  133,315건 ( 31.16%)

호르무즈 관련성 점수:


,score,article_count
0,0,258596
1,1,8547
2,2,160732


...

,country,article_count
0,US,162506
1,IN,58009
2,unknown,28815
3,GB,20101
4,PK,9813
5,AU,8434
6,CA,6512
7,ES,6450
8,NG,6328
9,TR,6013



NewsCatcher word_count 요약:


,nc_word_count
count,427875.000000
mean,721.210094
std,1283.457104
min,1.000000
25%,246.000000
50%,472.000000
75%,811.000000
max,104343.000000



paid_content:


,paid_content,article_count
0,False,409446
1,True,18429


## 10. 본문 추출 후보 만들기

본문 추출 후보는 **제목 중복으로 줄이지 않는다.**

기본 후보 조건은 연구 기간 안의 기사이고 URL이 정상인 경우다.  
`hormuz_relevance_score`, aggregator 플래그, `nc_word_count`, `paid_content` 등은 추출 우선순위와 사후 분석에 활용한다.

즉, 점수 0이라고 삭제하지 않는다.


In [9]:
def valid_http_url(url):
    try:
        p = urlsplit(str(url))
        return p.scheme in {"http", "https"} and bool(p.netloc)
    except Exception:
        return False


article_index["valid_http_url"] = article_index["url"].apply(valid_http_url)

extraction_candidates = article_index[
    article_index["in_research_period"] &
    article_index["valid_http_url"]
].copy().reset_index(drop=True)

# 추출 우선순위용 플래그
extraction_candidates["extraction_priority"] = (
    extraction_candidates["hormuz_relevance_score"]
    .map({2: 0, 1: 1, 0: 2})
    .fillna(2)
)

print("=" * 70)
print("EXTRACTION CANDIDATES")
print("=" * 70)
print(f"본문 추출 후보: {len(extraction_candidates):,}")
print("제목 중복 때문에 삭제된 기사: 0")

display(
    extraction_candidates[
        [
            "article_id", "title", "domain",
            "hormuz_relevance_score",
            "flag_aggregator_review",
            "nc_word_count",
            "nc_paid_content",
        ]
    ].head(10)
)


EXTRACTION CANDIDATES
본문 추출 후보: 427,875
제목 중복 때문에 삭제된 기사: 0


,article_id,title,domain,hormuz_relevance_score,flag_aggregator_review,nc_word_count,nc_paid_content
0,79481842df48c7b7,Iran: Strait of Hormuz Stays Shut Until Washin...,247ureports.com,2,False,553,False
1,72bc48a7d7b94b69,长江鱼类'超级密码本'登上《ADE》国际新刊创刊号,3g.cjn.cn,0,False,4264,False
2,c21d084e292bbe6e,"Trump says U.S. had ""very good"" talks with Ira...",89.218.85.198,0,False,127,False
3,6382d01eaf443d1a,«Improvisation» (Paco de Lucía e Jan Akkerman),abencerragem.blogspot.com,2,False,32,False
4,920b4bcd9ed129af,UNDP urges countries to broaden fiscal space &...,absafricatv.com,0,False,1109,False
5,c3bdc4c7171f10cc,FG raises the alarm over Red Sea oil supply di...,absafricatv9491.live-website.com,0,False,774,False
6,6857bbd8822284ba,"Becket Adams: CNN was right about the war, and...",ad-orientem.blogspot.com,0,False,230,False
7,05f932e07fd84711,America is Losing its Credibility With Both Fr...,ad-orientem.blogspot.com,0,False,628,False
8,e456e946b237a508,Iran sets out steep demands for reopening Hormuz,ad-orientem.blogspot.com,2,False,189,False
9,39b18d9f2fe308b5,OPEC+ debates oil output boost as war disrupts...,adaderana.lk,0,False,357,False


## 11. 추출기 벤치마크용 층화 표본

전체 수십만 건에 추출기를 먼저 돌리지 않는다.

추출기와 도메인이 강하게 얽히는 문제를 줄이기 위해 다음 축을 사용해 표본을 만든다.

- `hormuz_relevance_score`
- `flag_aggregator_review`
- `nc_country`
- 발행 월

각 층에서 최대 `PER_STRATUM`건을 고정 seed로 추출한다.

이 표본을 Fundus / news-please / trafilatura에 동일하게 적용하고, 일부는 사람이 직접 본문 품질을 확인한 뒤 기본 추출기와 fallback을 결정한다.


In [10]:
RANDOM_SEED = 2026
PER_STRATUM = 10
BENCHMARK_SAMPLE_SIZE_LIMIT = 300

benchmark_pool = extraction_candidates.copy()

benchmark_pool["published_dt"] = pd.to_datetime(
    benchmark_pool["published_date"], errors="coerce", utc=True
)
benchmark_pool["month"] = benchmark_pool["published_dt"].dt.to_period("M").astype(str)

benchmark_pool["country_stratum"] = (
    benchmark_pool["nc_country"]
    .fillna("(unknown)")
    .astype(str)
)

benchmark_pool["source_type_stratum"] = np.where(
    benchmark_pool["flag_aggregator_review"],
    "aggregator_review",
    "other"
)

strata_cols = [
    "hormuz_relevance_score",
    "source_type_stratum",
    "country_stratum",
    "month",
]

parts = []

for _, group in benchmark_pool.groupby(strata_cols, dropna=False):
    parts.append(
        group.sample(
            n=min(PER_STRATUM, len(group)),
            random_state=RANDOM_SEED,
        )
    )

benchmark_sample = (
    pd.concat(parts, ignore_index=True)
    .drop_duplicates("article_id")
)

if len(benchmark_sample) > BENCHMARK_SAMPLE_SIZE_LIMIT:
    benchmark_sample = benchmark_sample.sample(
        n=BENCHMARK_SAMPLE_SIZE_LIMIT,
        random_state=RANDOM_SEED,
    ).reset_index(drop=True)

print("=" * 70)
print("EXTRACTOR BENCHMARK SAMPLE")
print("=" * 70)
print(f"표본 기사 수: {len(benchmark_sample):,}")

display(
    benchmark_sample[
        [
            "article_id", "title", "domain",
            "nc_country", "month",
            "hormuz_relevance_score",
            "flag_aggregator_review",
            "url",
        ]
    ].head(30)
)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11164\3260032443.py:10: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  benchmark_pool["month"] = benchmark_pool["published_dt"].dt.to_period("M").astype(str)


EXTRACTOR BENCHMARK SAMPLE
표본 기사 수: 300


,article_id,title,domain,nc_country,month,hormuz_relevance_score,flag_aggregator_review,url
0,5dd8a0cae55da4a3,FM Denies Trump's Claim That Iran Requested Ta...,alethonews.com,LT,2026-07,0,False,https://alethonews.com/2026/07/11/fm-denies-tr...
1,72e0315270420690,Pakistan repays $3.45 billion loan to UAE,www.ariananews.af,AF,2026-04,2,False,https://www.ariananews.af/pakistan-repays-3-45...
2,dd4d8c3356d659bc,Iran-US war latest: Trump says ‘praise be to A...,newswav.com,MY,2026-08,2,False,https://newswav.com/article/iran-us-war-latest...
3,2bb275998e28ed01,What would be the impact of a US attack on Iran?,www.aljazeera.com,QA,2026-02,0,False,https://www.aljazeera.com/opinions/2026/2/3/wh...
4,d3b3d41038840866,World must 'accelerate' clean energy transitio...,www.omanobserver.om,OM,2026-04,0,False,https://www.omanobserver.om/article/1188817/wo...
5,667fd3f84cf95a2a,US allows Iranian oil sales during Tehran talks,www.pakistantoday.com.pk,PK,2026-06,1,False,https://www.pakistantoday.com.pk/2026/06/23/us...
6,defc398fa1735e3d,UK Deploys Rapid Sentry Air Defense System to ...,www.israeldefense.co.il,IL,2026-04,0,False,https://www.israeldefense.co.il/en/node/68434
7,06ac40888587904c,Trucks Are Just Driving The Oil Across The Ara...,www.jalopnik.com,HU,2026-05,1,False,https://www.jalopnik.com/2173916/avoid-straigh...
8,ef01a6d9d7089527,World: US fire on Iran tankers sparks reprisal...,www.newvision.co.ug,UG,2026-05,0,False,https://www.newvision.co.ug/category/world/us-...
9,e29c9a2bed831607,Iran temporarily closes Strait of Hormuz amid ...,peopledaily.digital,KE,2026-02,2,False,https://peopledaily.digital/news/iran-temporar...


## 12. 본문 추출 결과 스키마

아직 여기서 전체 본문 추출을 실행하지 않는다.

추출 결과는 최소한 아래 정보를 남긴다.

- `article_id`
- `url`
- `extractor_used`
- `extraction_status`
- `failure_reason`
- `body`
- `body_length`
- `http_status`
- `final_url`
- `fetched_at`
- `attempts`
- `extractor_version`
- `extracted_title`
- `extracted_date`
- `language_detected`
- `body_sha1`
- `nc_word_count`
- `nc_country`
- `nc_paid_content`

`published_date`는 연구의 기준 날짜이므로 본문 추출기가 제공한 날짜로 덮어쓰지 않는다.


In [11]:
EXTRACTION_COLUMNS = [
    "article_id",
    "url",
    "extractor_used",
    "extraction_status",
    "failure_reason",
    "body",
    "body_length",
    "http_status",
    "final_url",
    "fetched_at",
    "attempts",
    "extractor_version",
    "extracted_title",
    "extracted_date",
    "language_detected",
    "body_sha1",
    "nc_word_count",
    "nc_country",
    "nc_paid_content",
]


def empty_extraction_record():
    return {col: None for col in EXTRACTION_COLUMNS}


print("본문 추출 결과 컬럼 수:", len(EXTRACTION_COLUMNS))
print(EXTRACTION_COLUMNS)


본문 추출 결과 컬럼 수: 19
['article_id', 'url', 'extractor_used', 'extraction_status', 'failure_reason', 'body', 'body_length', 'http_status', 'final_url', 'fetched_at', 'attempts', 'extractor_version', 'extracted_title', 'extracted_date', 'language_detected', 'body_sha1', 'nc_word_count', 'nc_country', 'nc_paid_content']


## 13. 추출기 설치/선택은 벤치마크 후 결정

권장 비교 대상:

1. `trafilatura` — 범용 HTML 본문 추출기
2. `news-please` — 뉴스 기사 추출
3. `fundus` — 지원되는 publisher에 강점이 있는 뉴스 파서

중요한 점은 **Fundus를 무조건 첫 번째 추출기로 가정하지 않는 것**이다. 현재 데이터의 주요 도메인과 Fundus 지원 범위가 맞지 않을 수 있으므로 동일 표본에서 비교한다.

아래 셀은 선택적으로 패키지를 설치한다.


In [ ]:
# 필요할 때만 실행
# %pip install -q trafilatura news-please fundus

## 14. 벤치마크 실행 전 체크리스트

전체 추출 전에 `benchmark_sample`의 URL을 대상으로 각 추출기를 비교한다.

사람이 직접 확인할 때는 최소한 다음을 기록한다.

- 실제 기사 본문이 맞는가
- 본문이 충분히 긴가
- 메뉴/쿠키/에러 페이지가 섞였는가
- 제목과 본문이 맞는가
- 날짜가 맞는가
- 언어가 영어인가
- 같은 기사/신디케이션인가
- paywall 또는 차단 상태인가

추출 성공률만으로 기본 추출기를 결정하지 않는다. **본문 품질과 도메인별 성공률을 함께 평가한다.**

### 추출 상태 권장값

`success / empty / http_error / blocked / timeout / paywall_suspected / non_article / dead_link`

### 주의

- 4xx/dead link는 반복 요청하지 않는다.
- 5xx/timeout 등 일시 오류만 제한적으로 exponential backoff 재시도한다.
- paywall이나 Cloudflare 등의 제한을 우회하지 않는다.


## 15. 현재 단계 산출물 저장

이 셀은 새 분석용 메타데이터 파일을 만든다.

기존 `01_newscatcher_raw.csv`는 절대 덮어쓰지 않는다.

생성 파일:

- `05_article_index_v2.csv` — URL 중복 제거 후 전체 메타데이터
- `05_extraction_candidates_v2.csv` — 연구 기간 내 본문 추출 후보
- `05_extractor_benchmark_sample.csv` — 추출기 비교용 층화 표본

기존 `02_articles.csv`, `03_pre_diffbot_*`, `04_diagnostic_*`, `articles_partial.csv`와 이름을 섞지 않기 위해 `v2`를 붙인다.


In [12]:
ARTICLE_INDEX_V2_FILE = DATA_DIR / "05_article_index_v2.csv"
EXTRACTION_CANDIDATES_V2_FILE = DATA_DIR / "05_extraction_candidates_v2.csv"
BENCHMARK_SAMPLE_FILE = DATA_DIR / "05_extractor_benchmark_sample.csv"

article_index.to_csv(
    ARTICLE_INDEX_V2_FILE,
    index=False,
    encoding="utf-8-sig",
)

extraction_candidates.to_csv(
    EXTRACTION_CANDIDATES_V2_FILE,
    index=False,
    encoding="utf-8-sig",
)

benchmark_sample.to_csv(
    BENCHMARK_SAMPLE_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료")
print("-", ARTICLE_INDEX_V2_FILE)
print("-", EXTRACTION_CANDIDATES_V2_FILE)
print("-", BENCHMARK_SAMPLE_FILE)


저장 완료
- c:\Users\황태하\Hormuz_Research\hormuz_2026_data\05_article_index_v2.csv
- c:\Users\황태하\Hormuz_Research\hormuz_2026_data\05_extraction_candidates_v2.csv
- c:\Users\황태하\Hormuz_Research\hormuz_2026_data\05_extractor_benchmark_sample.csv


# 현재 단계 완료 기준

이 노트북을 끝까지 실행하면 다음 상태가 된다.

**NewsCatcher 원본**  
→ **URL 중복 제거**  
→ **원본 메타데이터 보존**  
→ **제목 중복은 삭제하지 않고 그룹화**  
→ **메타데이터 검토 플래그 생성**  
→ **연구 기간 내 본문 추출 후보 생성**  
→ **추출기 벤치마크 표본 생성**

그 다음 단계는 이 표본을 이용해 실제 본문 추출기를 비교하고, 기본 추출기 + fallback 조합을 확정한 뒤 전체 후보에 본문 추출을 적용하는 것이다.

## 절대 하지 않는 것

- NewsCatcher API 재수집
- 제목 중복을 `keep="first"`로 강제 삭제
- `hormuz_relevance_score == 0` 기사 일괄 삭제
- `description` 대신 잘린 `content`를 description으로 취급
- Diffbot Free 플랜으로 38만 건 이상 전체 본문 수집
- 본문 추출기의 날짜로 연구 기준 `published_date` 덮어쓰기
- paywall/Cloudflare/robots 제한 우회


---

# 본문 추출 벤치마크 단계 (v2에서 추가된 부분)

> **이 셀 위쪽(셀 0~30)은 `hormuz_textminig.ipynb`의 코드와 출력을 한 글자도 바꾸지 않고 그대로 옮긴 것이다.**  
> 아래 16~27번 섹션만 새로 추가되었고, 위쪽 셀에서 만든 변수(`DATA_DIR`, `extraction_candidates`, `benchmark_sample`, `EXTRACTION_COLUMNS` 등)를 그대로 이어서 사용한다.  
> 커널을 재시작해서 위쪽 셀을 다시 실행하지 않은 경우에는 `05_*.csv` 파일에서 자동으로 불러온다.

## 이 단계에서 하는 일 / 하지 않는 일

**하는 일**

1. 벤치마크 표본(약 300건)의 URL을 **한 번만** 가져와서 HTML을 디스크에 저장한다.
2. **같은 HTML**에 `Fundus` / `news-please` / `trafilatura`를 모두 적용한다. (추출기 차이와 접속 실패를 분리하기 위함)
3. 접속 실패·차단·페이월 의심·추출 실패를 모두 기록한다. 성공한 기사만 남기지 않는다.
4. 임계값 없이 **진단 지표**만 계산한다. (본문 길이, NewsCatcher 단어 수 대비 비율, 제목 일치도, boilerplate 흔적, 추출기 간 일치도)
5. 사람이 직접 라벨링할 시트(`06_benchmark_label_sheet.csv`)를 만들고, 라벨링 결과를 읽어 추출기별 정확도를 신뢰구간과 함께 계산한다.

**하지 않는 일**

- 427,875건 전체 추출 (벤치마크 결과를 보고 기본 추출기를 정한 다음 단계에서 한다)
- 추출기 자동 선택 (성공률만으로 정하지 않는다. 최종 선택은 라벨링 결과를 보고 사람이 한다)
- paywall / Cloudflare / robots.txt 우회 (막히면 `blocked` 등으로 기록만 한다)
- 본문 품질을 이유로 한 기사 삭제 (이 단계에는 삭제가 없다)
- 추출기가 준 날짜로 `published_date` 덮어쓰기

## 표본 설계에서 알아둘 점

- 셀 22의 표본은 "층별 최대 10건을 뽑은 뒤 300건을 무작위로 줄이는" 방식이라 **기사마다 뽑힐 확률이 다르다.** 그대로 평균을 내면 코퍼스 전체의 추출 성공률이 아니다. 그래서 층 크기로 계산한 **표본 가중치(`sample_weight`)** 를 붙여 가중 성공률을 함께 보고한다.
- Fundus 지원 도메인은 후보 전체의 약 3% 수준이라 300건 표본에는 몇 건 안 들어온다. 그래서 **Fundus 지원 도메인 보충 표본**(`sample_origin = fundus_supplement`)을 따로 더한다. 이 보충 표본은 "Fundus가 지원 매체에서 실제로 더 나은가"를 비교하는 용도이며, **코퍼스 전체 비율을 추정할 때는 사용하지 않는다.** (`FUNDUS_SUPPLEMENT_N = 0`이면 건너뛴다.)

## 윤리/운영 메모

- 요청 헤더의 User-Agent에 연구용 봇임을 밝힌다. `.env`에 `BENCH_CONTACT_EMAIL=내메일`을 넣어두면 헤더에 연락처가 들어간다.
- 같은 도메인에는 일정 간격을 두고 접속하고, robots.txt를 확인한다.
- 저장되는 원본 HTML(`benchmark_html_cache/`)은 저작권이 있는 자료이므로 **Git에 올리지 않는다.** `.gitignore`에 `hormuz_2026_data/benchmark_html_cache/`를 추가하는 것을 권장한다.
- robots.txt와 각 사이트 이용약관의 법적 해석은 소속 기관(지도교수/연구윤리 담당)에 확인한다.

## 16. 패키지 설치 (최초 1회)

`fundus`는 최신 `curl_cffi`(0.12 이상)와 함께 설치하면 `import` 단계에서 `normalize_browser_type` 오류가 나는 것이 확인되었다. 그래서 `curl_cffi`를 0.12 미만으로 고정해서 같이 설치한다.

설치 후 `import` 오류가 나면 **커널을 재시작**하고 이 섹션부터 다시 실행하면 된다. (위쪽 변수는 `05_*.csv`에서 자동 복구된다.)

In [1]:
# 최초 1회만 실행. 이미 설치했다면 건너뛰어도 된다.
%pip install -q "curl_cffi>=0.9,<0.12" fundus trafilatura news-please

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 17. 벤치마크 설정과 표본 불러오기

- 아래 숫자들(`FETCH_TIMEOUT`, `PER_DOMAIN_MIN_INTERVAL_SEC` 등)은 **운영 설정**이지 연구 판단 기준이 아니다. 본문 품질이나 기사 포함/제외 기준으로 쓰지 않는다.
- `BENCH_RUN_LIMIT`을 `20` 같은 작은 값으로 두면 시험 실행(smoke test)만 할 수 있다. 처음에는 20건으로 확인한 뒤 `None`으로 바꿔 전체 표본을 실행하는 것을 권장한다.
- 다시 실행하면 최종 상태로 기록된 URL은 건너뛰고, **일시적 실패는 자동으로 재시도**한다(`06_benchmark_fetch_log.jsonl` 기준). 전부 새로 가져오려면 `BENCH_REFETCH = True`.

In [2]:
import codecs
import json
import logging
import os
import re
import threading
import time
import importlib.metadata as importlib_metadata
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from urllib import robotparser
from urllib.parse import urlsplit

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()  # 프로젝트 폴더의 .env가 있으면 읽는다 (예: BENCH_CONTACT_EMAIL)

# ------------------------------------------------------------------
# 경로: 위쪽 셀의 DATA_DIR이 있으면 그대로 사용, 없으면(커널 재시작) 다시 찾는다.
# ------------------------------------------------------------------
if "DATA_DIR" not in globals():
    _found = None
    for _root in [Path.cwd()] + list(Path.cwd().parents):
        if (_root / "hormuz_2026_data").is_dir():
            _found = _root / "hormuz_2026_data"
            break
    if _found is None:
        raise FileNotFoundError("hormuz_2026_data 폴더를 찾지 못했습니다. Hormuz_Research 폴더에서 노트북을 여세요.")
    DATA_DIR = _found

BENCH_SAMPLE_CSV = DATA_DIR / "05_extractor_benchmark_sample.csv"
CANDIDATES_CSV = DATA_DIR / "05_extraction_candidates_v2.csv"

BENCH_HTML_DIR = DATA_DIR / "benchmark_html_cache"
BENCH_FETCH_LOG_JSONL = DATA_DIR / "06_benchmark_fetch_log.jsonl"
BENCH_FETCH_LOG_CSV = DATA_DIR / "06_benchmark_fetch_log.csv"
BENCH_RESULTS_CSV = DATA_DIR / "06_benchmark_extraction_results.csv"
BENCH_AGREEMENT_CSV = DATA_DIR / "06_benchmark_pairwise_agreement.csv"
BENCH_LABEL_SHEET_CSV = DATA_DIR / "06_benchmark_label_sheet.csv"
BENCH_LABELED_CSV = DATA_DIR / "06_benchmark_label_sheet_labeled.csv"   # 사람이 라벨링해서 저장할 파일
BENCH_HTML_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 운영 설정 (연구 판단 기준이 아님)
# ------------------------------------------------------------------
BENCH_RUN_LIMIT = 20            # 예: 20 -> 시험 실행. None -> 표본 전체
BENCH_REFETCH = False             # True -> 이미 가져온 URL도 다시 가져옴
RESPECT_ROBOTS = True
FETCH_TIMEOUT = (10, 30)          # (연결, 읽기) 초
MAX_RETRIES = 2                   # 5xx / 429 / timeout / 연결 오류만 재시도
BACKOFF_BASE_SEC = 2.0            # 재시도 대기: 2, 4, 8 ... 초
PER_DOMAIN_MIN_INTERVAL_SEC = 2.0
MAX_WORKERS = 6
MAX_HTML_BYTES = 8_000_000
FUNDUS_SUPPLEMENT_N = 60          # 0 이면 Fundus 보충 표본을 만들지 않음
FUNDUS_SUPPLEMENT_PER_DOMAIN = 3  # 한 도메인이 보충 표본을 지배하지 않도록 하는 상한

BENCH_CONTACT = os.getenv("BENCH_CONTACT_EMAIL", "").strip()
ROBOTS_AGENT = "HormuzResearchBenchmark"
USER_AGENT = (
    f"{ROBOTS_AGENT}/0.1 (academic text-mining research; "
    + (f"contact: {BENCH_CONTACT})" if BENCH_CONTACT else "contact: not-set)")
)

# 셀 22에서 쓴 값이 있으면 그대로 사용 (표본 가중치 계산에 필요)
_seed = globals().get("RANDOM_SEED", 2026)
_per_stratum = globals().get("PER_STRATUM", 10)
_sample_limit = globals().get("BENCHMARK_SAMPLE_SIZE_LIMIT", 300)
_strata_cols = ["hormuz_relevance_score", "source_type_stratum", "country_stratum", "month"]

BENCH_KEEP_COLS = [
    "article_id", "url", "title", "domain", "published_date",
    "nc_country", "hormuz_relevance_score", "flag_aggregator_review",
    "nc_word_count", "nc_paid_content",
]


def add_strata(df):
    # 셀 22와 같은 방식으로 층 변수를 만든다.
    d = df.copy()
    published = pd.to_datetime(d["published_date"], errors="coerce", utc=True)
    d["month"] = published.dt.strftime("%Y-%m")
    d["country_stratum"] = d["nc_country"].fillna("(unknown)").astype(str)
    d["source_type_stratum"] = np.where(
        d["flag_aggregator_review"].astype(str).str.lower().isin({"true", "1"}),
        "aggregator_review", "other",
    )
    return d


# ------------------------------------------------------------------
# 표본 / 후보 불러오기 (메모리에 있으면 그대로, 없으면 CSV)
# ------------------------------------------------------------------
if "extraction_candidates" in globals():
    _cand = extraction_candidates[BENCH_KEEP_COLS].copy()
else:
    _cand = pd.read_csv(CANDIDATES_CSV, usecols=BENCH_KEEP_COLS, dtype={"article_id": str}, encoding="utf-8-sig", low_memory=False)

if "benchmark_sample" in globals():
    _main = benchmark_sample[BENCH_KEEP_COLS].copy()
else:
    _main = pd.read_csv(BENCH_SAMPLE_CSV, usecols=BENCH_KEEP_COLS, dtype={"article_id": str}, encoding="utf-8-sig", low_memory=False)

_cand = add_strata(_cand)
_main = add_strata(_main)

# ------------------------------------------------------------------
# 표본 가중치 (Horvitz-Thompson): 셀 22의 2단계 추출 확률의 역수
#   1단계: 층(h)에서 min(PER_STRATUM, N_h)건 -> 포함확률 n_h / N_h
#   2단계: 합집합 U가 한도를 넘으면 한도/U 비율로 무작위 축소
# ------------------------------------------------------------------
_stratum_n = _cand.groupby(_strata_cols, dropna=False).size().rename("N_h").reset_index()
_stratum_n["n_h"] = _stratum_n["N_h"].clip(upper=_per_stratum)
_union_size = int(_stratum_n["n_h"].sum())
_second_stage = (_sample_limit / _union_size) if _union_size > _sample_limit else 1.0

_main = _main.merge(_stratum_n, on=_strata_cols, how="left")
_main["sample_weight"] = (_main["N_h"] / _main["n_h"]) / _second_stage
_main["sample_origin"] = "stratified_main"

print("=" * 70)
print("BENCHMARK SETUP")
print("=" * 70)
print(f"후보 전체: {len(_cand):,}건 / 층 수: {len(_stratum_n):,} / 1단계 합집합: {_union_size:,}건 / 2단계 비율: {_second_stage:.4f}")
print(f"주 표본: {len(_main):,}건")
print(f"가중치 합계: {_main['sample_weight'].sum():,.0f}  (후보 전체 {len(_cand):,}건과 비슷해야 정상)")
print("User-Agent:", USER_AGENT)
if not BENCH_CONTACT:
    print("(참고) .env에 BENCH_CONTACT_EMAIL이 없어 연락처 없이 실행됩니다.")

c:\Users\황태하\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BENCHMARK SETUP
후보 전체: 427,875건 / 층 수: 2,776 / 1단계 합집합: 18,969건 / 2단계 비율: 0.0158
주 표본: 300건
가중치 합계: 551,252  (후보 전체 427,875건과 비슷해야 정상)
User-Agent: HormuzResearchBenchmark/0.1 (academic text-mining research; contact: not-set)
(참고) .env에 BENCH_CONTACT_EMAIL이 없어 연락처 없이 실행됩니다.


## 18. Fundus 지원 도메인 확인과 보충 표본

Fundus는 **퍼블리셔별 파서가 있는 매체에서만** 동작한다. 설치된 Fundus 버전이 지원하는 도메인을 직접 읽어서 후보 기사가 얼마나 해당하는지 확인한다. (감사에서 문자열 대조로 본 약 2.9%는 규모 추정이었고, 여기서는 설치된 버전 기준으로 다시 센다.)

In [3]:
def build_fundus_host_map():
    host_map = {}
    try:
        from fundus import PublisherCollection
    except Exception as exc:
        print("[경고] Fundus를 불러오지 못했습니다:", repr(exc)[:200])
        print("       섹션 16의 설치 셀을 실행했는지, curl_cffi 버전이 0.12 미만인지 확인하세요.")
        return host_map

    for group_name in dir(PublisherCollection):
        if group_name.startswith("_"):
            continue
        group = getattr(PublisherCollection, group_name)
        try:
            members = list(group)
        except TypeError:
            continue
        for pub in members:
            if getattr(pub, "deprecated", False):
                continue
            host = urlsplit(pub.domain).netloc.lower()
            host = host[4:] if host.startswith("www.") else host
            host_map[host] = pub
    return host_map


FUNDUS_HOST_MAP = build_fundus_host_map()


def fundus_publisher_for(host):
    # 감사 때와 같은 기준: www.를 뗀 호스트가 정확히 일치할 때만 지원으로 본다.
    host = str(host).lower().split(":")[0]
    host = host[4:] if host.startswith("www.") else host
    return FUNDUS_HOST_MAP.get(host)


_cand["fundus_supported"] = _cand["domain"].apply(lambda d: fundus_publisher_for(d) is not None)
_main["fundus_supported"] = _main["domain"].apply(lambda d: fundus_publisher_for(d) is not None)

print("=" * 70)
print("FUNDUS COVERAGE")
print("=" * 70)
print(f"Fundus 지원 퍼블리셔 수(설치 버전 기준): {len(FUNDUS_HOST_MAP):,}")
print(f"후보 중 지원 도메인 기사: {_cand['fundus_supported'].sum():,}건 ({_cand['fundus_supported'].mean() * 100:.2f}%)")
print(f"후보 중 지원 도메인 수: {_cand.loc[_cand['fundus_supported'], 'domain'].nunique():,}")
print(f"주 표본({len(_main)}건) 중 지원 도메인 기사: {_main['fundus_supported'].sum():,}건")

# ------------------------------------------------------------------
# Fundus 보충 표본: 주 표본에 지원 매체가 거의 없으므로 따로 더한다.
# 코퍼스 전체 비율 추정에는 쓰지 않는다. (sample_weight = NaN)
# ------------------------------------------------------------------
_supplement = _main.iloc[0:0].copy()
if FUNDUS_SUPPLEMENT_N and len(FUNDUS_HOST_MAP):
    _pool = _cand[_cand["fundus_supported"] & ~_cand["article_id"].isin(_main["article_id"])]
    _pool = _pool.sample(frac=1.0, random_state=_seed)
    _pool = _pool.groupby("domain", sort=False).head(FUNDUS_SUPPLEMENT_PER_DOMAIN)
    _supplement = _pool.sample(n=min(FUNDUS_SUPPLEMENT_N, len(_pool)), random_state=_seed).copy()
    _supplement["sample_weight"] = np.nan
    _supplement["sample_origin"] = "fundus_supplement"
    print(f"Fundus 보충 표본: {len(_supplement):,}건 / 도메인 {_supplement['domain'].nunique():,}개")

bench_run = pd.concat([_main, _supplement], ignore_index=True)
bench_run = bench_run.drop(columns=[c for c in ["N_h", "n_h"] if c in bench_run.columns])
bench_run["nc_word_count"] = pd.to_numeric(bench_run["nc_word_count"], errors="coerce")
bench_run["host"] = bench_run["url"].apply(lambda u: urlsplit(str(u)).netloc.lower().split(":")[0])

if BENCH_RUN_LIMIT:
    bench_run = bench_run.sample(n=min(BENCH_RUN_LIMIT, len(bench_run)), random_state=_seed).reset_index(drop=True)
    print(f"[주의] BENCH_RUN_LIMIT={BENCH_RUN_LIMIT} -> 시험 실행 중입니다. 전체 실행은 None으로 바꾸세요.")

print(f"\n이번 실행 대상: {len(bench_run):,}건")
display(bench_run["sample_origin"].value_counts().rename_axis("sample_origin").reset_index(name="article_count"))

FUNDUS COVERAGE
Fundus 지원 퍼블리셔 수(설치 버전 기준): 173
후보 중 지원 도메인 기사: 11,790건 (2.76%)
후보 중 지원 도메인 수: 72
주 표본(300건) 중 지원 도메인 기사: 4건
Fundus 보충 표본: 60건 / 도메인 44개
[주의] BENCH_RUN_LIMIT=20 -> 시험 실행 중입니다. 전체 실행은 None으로 바꾸세요.

이번 실행 대상: 20건


,sample_origin,article_count
0,stratified_main,15
1,fundus_supplement,5


## 19. URL 가져오기 (한 번만 가져와서 HTML 저장)

세 추출기가 **같은 HTML**을 받도록, 먼저 URL을 한 번만 가져와서 `benchmark_html_cache/{article_id}.html`로 저장한다. 이렇게 하면

- 접속 실패(차단, 404, timeout)와 추출기 성능 차이를 구분할 수 있고,
- 나중에 추출기 버전을 바꿔도 다시 접속하지 않고 같은 HTML로 재현할 수 있다.

처리 원칙

| 상황 | 처리 | 재시도 |
|---|---|---|
| 404 / 410 | `dead_link` | 안 함 |
| 401 / 403 / 451 / 429(재시도 후) / Cloudflare 확인 페이지 | `blocked` (우회하지 않음) | 429만 제한적으로 |
| 402 | `paywall_suspected` | 안 함 |
| 5xx / timeout / 일시적 연결 오류 | `http_error` / `timeout` | 지수 백오프, 최대 `MAX_RETRIES`회 |
| HTML이 아닌 응답, 홈페이지로 리다이렉트 | `non_article` | 안 함 |
| robots.txt 금지 / 확인 불가 | `blocked` (`robots_disallowed` / `robots_unreachable`) | 안 함 |

모든 시도 결과는 `06_benchmark_fetch_log.jsonl`에 한 줄씩 즉시 기록된다.

**다시 실행할 때의 동작 (이어서 실행 + 일시적 실패 자동 재시도)**

- 기록이 없는 URL은 새로 가져온다. (중간에 중단되어도 이어서 실행된다)
- **일시적 실패는 다음 실행에서 자동으로 재시도한다:** `timeout`, `http_error`, `blocked` 중 `http_429` / `robots_unreachable`, `dead_link` 중 `dns_failure`
- **최종 상태는 다시 가져오지 않는다:** `ok`, `dead_link`(404/410), `blocked`(403, Cloudflare 확인, robots 금지), `paywall_suspected`, `non_article`
- 재시도 결과는 같은 `article_id`의 새 기록으로 쌓이고, 분석에는 **가장 마지막 기록**만 쓴다.
- 최종 상태까지 전부 처음부터 다시 받으려면 `BENCH_REFETCH = True`.

In [4]:
FETCH_STATUS_VALUES = ["ok", "dead_link", "blocked", "paywall_suspected", "http_error", "timeout", "non_article"]

CHALLENGE_TITLE_RE = re.compile(
    r"just a moment|attention required|access denied|are you a robot|verify you are human|"
    r"security check|captcha|ddos protection|pardon our interruption|checking your browser|request blocked",
    re.I,
)

_thread_local = threading.local()


def get_session():
    session = getattr(_thread_local, "session", None)
    if session is None:
        session = requests.Session()
        session.headers.update({
            "User-Agent": USER_AGENT,
            "Accept": "text/html,application/xhtml+xml;q=0.9,*/*;q=0.5",
            "Accept-Language": "en-US,en;q=0.8",
        })
        session.max_redirects = 5
        _thread_local.session = session
    return session


class DomainThrottle:
    # 같은 호스트에는 최소 간격을 두고 접속한다. (스레드 안전)
    def __init__(self, min_interval):
        self.min_interval = min_interval
        self._lock = threading.Lock()
        self._next_slot = {}

    def wait(self, host):
        with self._lock:
            now = time.monotonic()
            start = max(now, self._next_slot.get(host, 0.0))
            self._next_slot[host] = start + self.min_interval
        delay = start - now
        if delay > 0:
            time.sleep(delay)


class RobotsCache:
    def __init__(self):
        self._lock = threading.Lock()
        self._cache = {}

    def _load(self, origin):
        try:
            resp = get_session().get(origin + "/robots.txt", timeout=(5, 10))
        except requests.exceptions.RequestException:
            return ("unreachable", None)
        if resp.status_code == 200:
            parser = robotparser.RobotFileParser()
            parser.parse(resp.text.splitlines())
            parser.modified()
            return ("rules", parser)
        if 400 <= resp.status_code < 500:
            return ("allow_all", None)       # robots.txt 없음(4xx) -> 제한 없음으로 해석
        return ("unreachable", None)         # 5xx 등 -> 확인 불가이므로 가져오지 않음

    def allowed(self, url):
        parts = urlsplit(url)
        origin = f"{parts.scheme}://{parts.netloc}"
        with self._lock:
            entry = self._cache.get(origin)
        if entry is None:
            entry = self._load(origin)
            with self._lock:
                self._cache[origin] = entry
        kind, parser = entry
        if kind == "allow_all":
            return True, "robots_not_found"
        if kind == "unreachable":
            return False, "robots_unreachable"
        ok = parser.can_fetch(ROBOTS_AGENT, url)
        return ok, (None if ok else "robots_disallowed")


THROTTLE = DomainThrottle(PER_DOMAIN_MIN_INTERVAL_SEC)
ROBOTS = RobotsCache()
_log_lock = threading.Lock()


def utc_now_iso():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def detect_encoding(content, content_type):
    # 1) BOM 2) HTTP 헤더에 명시된 charset 3) <meta charset> 4) utf-8 5) 통계적 추정
    if content.startswith(codecs.BOM_UTF8):
        return "utf-8-sig"
    candidates = []
    match = re.search(r"charset=([\w\-]+)", content_type or "", re.I)
    if match:
        candidates.append(match.group(1))
    head = content[:4096].decode("ascii", errors="ignore")
    match = re.search(r"<meta[^>]+charset=[\"']?([\w\-]+)", head, re.I)
    if match:
        candidates.append(match.group(1))
    candidates.append("utf-8")
    for enc in candidates:
        try:
            codecs.lookup(enc)
            content.decode(enc)
            return enc
        except (LookupError, UnicodeDecodeError):
            continue
    try:
        from charset_normalizer import from_bytes
        best = from_bytes(content[:200000]).best()
        if best is not None and best.encoding:
            return best.encoding
    except Exception:
        pass
    return "utf-8"


def decode_html(content, encoding):
    return content.decode(encoding or "utf-8", errors="replace")


def _retry_after_seconds(resp):
    value = resp.headers.get("Retry-After", "")
    return min(int(value), 60) if value.isdigit() else None


def fetch_one(article_id, url):
    rec = {
        "article_id": article_id, "url": url,
        "fetch_status": None, "failure_reason": None,
        "http_status": None, "final_url": None, "fetched_at": None, "attempts": 0,
        "html_path": None, "html_sha1": None, "html_bytes": None,
        "content_type": None, "html_encoding": None, "redirect_count": None,
        "robots_note": None, "html_truncated": False, "redirected_to_root": False,
    }
    host = urlsplit(url).netloc.lower()

    if RESPECT_ROBOTS:
        allowed, note = ROBOTS.allowed(url)
        rec["robots_note"] = note
        if not allowed:
            rec.update(fetch_status="blocked", failure_reason=note, fetched_at=utc_now_iso())
            return rec

    import hashlib
    for attempt in range(1, MAX_RETRIES + 2):
        rec["attempts"] = attempt
        is_last = attempt == MAX_RETRIES + 1
        backoff = BACKOFF_BASE_SEC * (2 ** (attempt - 1))
        THROTTLE.wait(host)
        rec["fetched_at"] = utc_now_iso()

        try:
            resp = get_session().get(url, timeout=FETCH_TIMEOUT, stream=True, allow_redirects=True)
            chunks, total, truncated = [], 0, False
            for chunk in resp.iter_content(65536):
                chunks.append(chunk)
                total += len(chunk)
                if total > MAX_HTML_BYTES:
                    truncated = True
                    break
            content = b"".join(chunks)
            resp.close()
        except requests.exceptions.Timeout:
            if is_last:
                rec.update(fetch_status="timeout", failure_reason="timeout")
                return rec
            time.sleep(backoff)
            continue
        except requests.exceptions.SSLError as exc:
            rec.update(fetch_status="http_error", failure_reason="ssl_error:" + str(exc)[:80])
            return rec
        except requests.exceptions.TooManyRedirects:
            rec.update(fetch_status="http_error", failure_reason="too_many_redirects")
            return rec
        except requests.exceptions.ConnectionError as exc:
            text = str(exc)
            if re.search(r"Read timed out|ReadTimeout", text):
                if is_last:
                    rec.update(fetch_status="timeout", failure_reason="read_timeout")
                    return rec
                time.sleep(backoff)
                continue
            if re.search(r"getaddrinfo failed|Name or service not known|NameResolution|nodename nor servname", text):
                rec.update(fetch_status="dead_link", failure_reason="dns_failure")
                return rec
            if is_last:
                rec.update(fetch_status="http_error", failure_reason="connection_error:" + text[:80])
                return rec
            time.sleep(backoff)
            continue
        except requests.exceptions.RequestException as exc:
            rec.update(fetch_status="http_error", failure_reason=f"request_error:{type(exc).__name__}")
            return rec

        status = resp.status_code
        ctype = resp.headers.get("Content-Type", "")
        rec.update(
            http_status=status, final_url=resp.url, content_type=ctype[:100],
            redirect_count=len(resp.history), html_truncated=truncated,
        )

        if resp.headers.get("cf-mitigated", "").lower() == "challenge":
            rec.update(fetch_status="blocked", failure_reason="cloudflare_challenge")
            return rec
        if status in (404, 410):
            rec.update(fetch_status="dead_link", failure_reason=f"http_{status}")
            return rec
        if status == 429 or status >= 500:
            if is_last:
                if status == 429:
                    rec.update(fetch_status="blocked", failure_reason="http_429")
                else:
                    rec.update(fetch_status="http_error", failure_reason=f"http_{status}")
                return rec
            time.sleep(max(backoff, _retry_after_seconds(resp) or 0))
            continue
        if status == 402:
            rec.update(fetch_status="paywall_suspected", failure_reason="http_402")
            return rec
        if status in (401, 403, 451):
            rec.update(fetch_status="blocked", failure_reason=f"http_{status}")
            return rec
        if status >= 400 or not (200 <= status < 300):
            rec.update(fetch_status="http_error", failure_reason=f"http_{status}")
            return rec

        # ---- 2xx ----
        if ctype and "html" not in ctype.lower():
            rec.update(fetch_status="non_article", failure_reason="content_type:" + ctype[:40])
            return rec

        encoding = detect_encoding(content, ctype)
        html_text = decode_html(content, encoding)
        title_match = re.search(r"<title[^>]*>(.*?)</title>", html_text[:50000], re.I | re.S)
        if title_match and CHALLENGE_TITLE_RE.search(title_match.group(1)):
            rec.update(fetch_status="blocked", failure_reason="bot_challenge_title")
            return rec

        path = BENCH_HTML_DIR / f"{article_id}.html"
        path.write_bytes(content)
        rec.update(
            html_path=str(path), html_sha1=hashlib.sha1(content).hexdigest(),
            html_bytes=len(content), html_encoding=encoding,
        )
        final_path = urlsplit(resp.url).path.strip("/")
        orig_path = urlsplit(url).path.strip("/")
        if final_path == "" and orig_path != "":
            rec.update(fetch_status="non_article", failure_reason="redirected_to_homepage", redirected_to_root=True)
        else:
            rec.update(fetch_status="ok", failure_reason=None)
        return rec

    return rec


def load_fetch_log():
    if not BENCH_FETCH_LOG_JSONL.exists():
        return pd.DataFrame()
    rows = []
    with open(BENCH_FETCH_LOG_JSONL, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    # 같은 article_id가 여러 번 기록된 경우(BENCH_REFETCH) 마지막 기록만 사용
    return df.drop_duplicates("article_id", keep="last").reset_index(drop=True) if len(df) else df


def is_retryable(rec):
    # 일시적일 수 있는 실패만 True. 최종 상태(ok, dead_link, blocked, paywall_suspected, non_article)는 False.
    status = rec.get("fetch_status")
    reason = str(rec.get("failure_reason") or "")
    if status in {"timeout", "http_error"}:
        return True
    if status == "blocked" and reason in {"http_429", "robots_unreachable"}:
        return True          # 시간이 지나면 풀릴 수 있는 차단 (우회가 아니라 재시도)
    if status == "dead_link" and reason == "dns_failure":
        return True          # 순간적인 네트워크/DNS 장애일 수 있음
    return False


def run_fetch(run_df):
    log = load_fetch_log()
    if BENCH_REFETCH or log.empty:
        done = set()
        n_retry = 0
    else:
        retry_ids = set(log.loc[log.apply(is_retryable, axis=1), "article_id"])
        done = set(log["article_id"]) - retry_ids
        n_retry = len(retry_ids & set(run_df["article_id"]))
    todo = run_df[~run_df["article_id"].isin(done)]
    n_new = len(todo) - n_retry
    print(f"가져올 URL: {len(todo):,}건 (처음 가져오는 {n_new:,}건 + 이전 일시적 실패 재시도 {n_retry:,}건) / 최종 상태로 건너뜀 {len(run_df) - len(todo):,}건")
    if todo.empty:
        return

    def worker(article_id, url):
        try:
            return fetch_one(article_id, url)
        except Exception as exc:  # 예상 못 한 오류도 기록으로 남긴다
            return {"article_id": article_id, "url": url, "fetch_status": "http_error",
                    "failure_reason": f"unexpected:{type(exc).__name__}:{str(exc)[:80]}",
                    "fetched_at": utc_now_iso(), "attempts": 0}

    with open(BENCH_FETCH_LOG_JSONL, "a", encoding="utf-8") as log_file:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = [pool.submit(worker, r.article_id, r.url) for r in todo.itertuples()]
            for fut in tqdm(as_completed(futures), total=len(futures), desc="fetch"):
                rec = fut.result()
                with _log_lock:
                    log_file.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")
                    log_file.flush()


print("가져오기 함수 준비 완료 (아직 실행하지 않음)")

가져오기 함수 준비 완료 (아직 실행하지 않음)


## 20. 가져오기 실행

처음에는 `BENCH_RUN_LIMIT = 20`으로 시험 실행하는 것을 권장한다. 이 셀은 최종 상태로 기록된 URL만 건너뛰고, **일시적 실패(timeout, http_error, 429 등)는 다시 실행할 때 자동으로 재시도**한다. 중단 후 다시 실행해도 된다.

In [5]:
run_fetch(bench_run)

fetch_log_df = load_fetch_log()
fetch_log_df = fetch_log_df[fetch_log_df["article_id"].isin(bench_run["article_id"])].copy()

print("\n가져오기 상태:")
display(fetch_log_df["fetch_status"].value_counts(dropna=False).rename_axis("fetch_status").reset_index(name="article_count"))
print("실패 사유 상위:")
display(
    fetch_log_df.loc[fetch_log_df["fetch_status"] != "ok", "failure_reason"]
    .fillna("(none)").value_counts().head(15).rename_axis("failure_reason").reset_index(name="article_count")
)

가져올 URL: 20건 (처음 가져오는 20건 + 이전 일시적 실패 재시도 0건) / 최종 상태로 건너뜀 0건


fetch: 100%|██████████| 20/20 [00:09<00:00,  2.00it/s]


가져오기 상태:


,fetch_status,article_count
0,ok,17
1,blocked,2
2,dead_link,1


실패 사유 상위:


,failure_reason,article_count
0,cloudflare_challenge,2
1,http_404,1


## 21. 추출기 래퍼

세 추출기를 **같은 입력(HTML 문자열, URL)** 으로 호출하고 같은 형태의 결과를 돌려주는 얇은 래퍼만 둔다. 추출기 옵션은 기본값을 최대한 유지한다. (벤치마크 단계에서 옵션을 튜닝하면 비교가 불공정해진다.)

| 추출기 | 호출 | 비고 |
|---|---|---|
| `trafilatura` | `extract(..., output_format="json", with_metadata=True, include_comments=False)` | 범용. 댓글만 제외 |
| `news-please` | `NewsPlease.from_html(html, url=url, fetch_images=False)` | 이미지 다운로드(네트워크 요청)를 끈다 |
| `fundus` | 지원 퍼블리셔의 `parser().parse(html, error_handling="catch")` | 지원 도메인에서만 실행. 그 밖은 `unsupported_publisher` |

**`extraction_status`는 셀 27의 권장값을 따르되, 벤치마크용으로 두 값을 추가한다.**

`success / empty / http_error / blocked / timeout / paywall_suspected / non_article / dead_link` + `unsupported_publisher`(Fundus가 해당 도메인을 지원하지 않음) + `extractor_error`(추출기 내부 예외)

여기서 **`success`는 "본문 텍스트가 비어 있지 않다"는 뜻일 뿐 본문이 맞다는 뜻이 아니다.** 본문 품질은 섹션 23~26에서 따로 본다.

In [6]:
from langdetect import DetectorFactory
import hashlib

DetectorFactory.seed = 0  # 언어 판별 결과를 재현 가능하게 고정

for _name in ("trafilatura", "htmldate", "courlan", "newspaper", "fundus", "readability", "urllib3"):
    logging.getLogger(_name).setLevel(logging.ERROR)

if "detect_language_safe" not in globals():
    from langdetect import detect, LangDetectException

    def detect_language_safe(text):
        text = str(text or "").strip()
        if len(text) < 50:
            return None
        try:
            return detect(text)
        except LangDetectException:
            return None

if "normalize_title" not in globals():
    def normalize_title(text):
        if pd.isna(text):
            return ""
        text = str(text).lower().strip()
        text = re.sub(r"https?://\S+", " ", text)
        text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
        return re.sub(r"\s+", " ", text).strip()

if "EXTRACTION_COLUMNS" not in globals():
    EXTRACTION_COLUMNS = [
        "article_id", "url", "extractor_used", "extraction_status", "failure_reason", "body", "body_length",
        "http_status", "final_url", "fetched_at", "attempts", "extractor_version", "extracted_title",
        "extracted_date", "language_detected", "body_sha1", "nc_word_count", "nc_country", "nc_paid_content",
    ]

# 셀 24의 19개 컬럼을 그대로 쓰고, 벤치마크 분석용 컬럼만 뒤에 덧붙인다.
BENCH_EXTRA_COLUMNS = [
    "sample_origin", "sample_weight", "domain", "fundus_supported", "hormuz_relevance_score",
    "flag_aggregator_review", "month", "word_count_extracted", "extractor_runtime_sec",
    "extractor_note", "extractor_settings", "html_path",
]
BENCH_RESULT_COLUMNS = list(EXTRACTION_COLUMNS) + BENCH_EXTRA_COLUMNS

EXTRACTOR_NAMES = ["fundus", "news-please", "trafilatura"]
EXTRACTOR_PACKAGES = {"fundus": "fundus", "news-please": "news-please", "trafilatura": "trafilatura"}
EXTRACTOR_SETTINGS = {
    "fundus": "publisher parser (latest), error_handling=catch",
    "news-please": "from_html, fetch_images=False",
    "trafilatura": "output=json, with_metadata=True, include_comments=False, other=default",
}


def extractor_version(name):
    pkg = EXTRACTOR_PACKAGES[name]
    try:
        return f"{pkg}=={importlib_metadata.version(pkg)}"
    except Exception:
        return f"{pkg}==(not installed)"


def _iso(value):
    if value is None:
        return None
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return str(value) or None


def run_trafilatura(html_text, url, host):
    import trafilatura
    out = trafilatura.extract(
        html_text, url=url, output_format="json", with_metadata=True, include_comments=False,
    )
    if not out:
        return {}
    data = json.loads(out)
    return {"body": data.get("text") or data.get("raw_text"), "title": data.get("title"), "date": data.get("date")}


def run_newsplease(html_text, url, host):
    from newsplease import NewsPlease
    article = NewsPlease.from_html(html_text, url=url, fetch_images=False)
    if article is None:
        return {}
    return {"body": article.maintext, "title": article.title, "date": _iso(article.date_publish)}


def run_fundus(html_text, url, host):
    publisher = fundus_publisher_for(host)
    if publisher is None:
        return {"unsupported": True}
    parsed = publisher.parser().parse(html_text, error_handling="catch")
    body = parsed.get("body")
    body_error = body if isinstance(body, Exception) else None
    text = None if (body is None or body_error is not None) else (str(body) or None)
    title = parsed.get("title")
    free_access = parsed.get("free_access")
    return {
        "body": text,
        "title": title if isinstance(title, str) else None,
        "date": _iso(parsed.get("publishing_date")) if not isinstance(parsed.get("publishing_date"), Exception) else None,
        "note": "free_access_false" if free_access is False else None,
        "error": f"body_parse_error:{type(body_error).__name__}" if body_error is not None else None,
    }


EXTRACTOR_FUNCS = {"fundus": run_fundus, "news-please": run_newsplease, "trafilatura": run_trafilatura}


def normalize_body_for_hash(text):
    return re.sub(r"\s+", " ", text).strip()


def run_extractor(name, html_text, url, host):
    started = time.perf_counter()
    try:
        result = EXTRACTOR_FUNCS[name](html_text, url, host)
    except Exception as exc:
        result = {"error": f"{type(exc).__name__}: {str(exc)[:150]}"}
    result["runtime"] = time.perf_counter() - started

    body = (result.get("body") or "").strip()
    if result.get("unsupported"):
        status, reason = "unsupported_publisher", "no_fundus_parser_for_domain"
    elif result.get("error") and not body:
        status, reason = "extractor_error", result["error"]
    elif body:
        status, reason = "success", None
    elif result.get("note") == "free_access_false":
        status, reason = "paywall_suspected", "fundus_free_access_false"
    else:
        status, reason = "empty", "no_text_extracted"
    result.update(body=body or None, status=status, reason=reason)
    return result


def build_result_record(meta, fetch_rec, name, result):
    body = result.get("body") if result else None
    rec = {col: None for col in BENCH_RESULT_COLUMNS}
    rec.update({
        "article_id": meta["article_id"], "url": meta["url"], "extractor_used": name,
        "http_status": fetch_rec.get("http_status"), "final_url": fetch_rec.get("final_url"),
        "fetched_at": fetch_rec.get("fetched_at"), "attempts": fetch_rec.get("attempts"),
        "extractor_version": extractor_version(name), "extractor_settings": EXTRACTOR_SETTINGS[name],
        "nc_word_count": meta.get("nc_word_count"), "nc_country": meta.get("nc_country"),
        "nc_paid_content": meta.get("nc_paid_content"),
        "sample_origin": meta.get("sample_origin"), "sample_weight": meta.get("sample_weight"),
        "domain": meta.get("domain"), "fundus_supported": meta.get("fundus_supported"),
        "hormuz_relevance_score": meta.get("hormuz_relevance_score"),
        "flag_aggregator_review": meta.get("flag_aggregator_review"), "month": meta.get("month"),
        "html_path": fetch_rec.get("html_path"),
    })
    if result is None:
        # 가져오기 자체가 실패한 경우: 세 추출기 모두 같은 실패 상태로 기록해 분모를 맞춘다.
        rec.update(extraction_status=fetch_rec.get("fetch_status"), failure_reason=fetch_rec.get("failure_reason"))
        return rec
    rec.update(
        extraction_status=result["status"], failure_reason=result["reason"],
        extracted_title=result.get("title"), extracted_date=result.get("date"),
        extractor_note=result.get("note"), extractor_runtime_sec=round(result["runtime"], 4),
    )
    if body:
        rec.update(
            body=body, body_length=len(body), word_count_extracted=len(body.split()),
            language_detected=detect_language_safe(body[:5000]),
            body_sha1=hashlib.sha1(normalize_body_for_hash(body).encode("utf-8")).hexdigest(),
        )
    return rec


def run_extraction(run_df, fetch_df):
    fetch_by_id = fetch_df.set_index("article_id").to_dict("index") if len(fetch_df) else {}
    records = []
    for meta in tqdm(run_df.to_dict("records"), desc="extract"):
        fetch_rec = fetch_by_id.get(meta["article_id"])
        if fetch_rec is None:
            fetch_rec = {"fetch_status": "http_error", "failure_reason": "not_fetched"}
        fetch_ok = fetch_rec.get("fetch_status") == "ok" and fetch_rec.get("html_path")
        if fetch_ok:
            html_text = decode_html(Path(fetch_rec["html_path"]).read_bytes(), fetch_rec.get("html_encoding"))
        for name in EXTRACTOR_NAMES:
            if fetch_ok:
                result = run_extractor(name, html_text, meta["url"], meta["host"])
                records.append(build_result_record(meta, fetch_rec, name, result))
            else:
                records.append(build_result_record(meta, fetch_rec, name, None))
    return pd.DataFrame(records, columns=BENCH_RESULT_COLUMNS)


print("추출기 버전:")
for _n in EXTRACTOR_NAMES:
    print(" -", _n, "->", extractor_version(_n))

추출기 버전:
 - fundus -> fundus==0.5.7
 - news-please -> news-please==1.6.16
 - trafilatura -> trafilatura==2.0.0


## 22. 추출 실행

저장된 HTML에서 세 추출기를 실행한다. 이 단계는 네트워크를 쓰지 않으므로 몇 번이든 다시 실행해도 된다. (추출기 버전을 올려서 다시 비교할 때도 같은 HTML을 쓸 수 있다.)

결과는 **추출기 × 기사** 긴 형식(long format)이다. 가져오기 실패 기사도 세 추출기 모두에 같은 실패 상태로 남겨서, 추출기별 성공률의 분모가 같도록 한다.

In [7]:
benchmark_results = run_extraction(bench_run, fetch_log_df)

print("=" * 70)
print("EXTRACTION RESULTS (long format)")
print("=" * 70)
print(f"행 수: {len(benchmark_results):,} (= 기사 {benchmark_results['article_id'].nunique():,} x 추출기 {len(EXTRACTOR_NAMES)})")
display(
    pd.crosstab(benchmark_results["extractor_used"], benchmark_results["extraction_status"], margins=True)
)

extract: 100%|██████████| 20/20 [00:22<00:00,  1.13s/it]

EXTRACTION RESULTS (long format)
행 수: 60 (= 기사 20 x 추출기 3)


extraction_status,blocked,dead_link,empty,extractor_error,success,unsupported_publisher,All
extractor_used,,,,,,,
fundus,2,1,1,0,5,11,20
news-please,2,1,0,1,16,0,20
trafilatura,2,1,0,0,17,0,20
All,6,3,1,1,38,11,60


## 23. 자동 진단 지표 (삭제 기준이 아님)

본문을 직접 읽기 전에 **사람이 어디를 먼저 봐야 하는지** 알려주는 지표만 계산한다. 어느 값도 기사 포함/제외의 임계값으로 쓰지 않는다. (임계값은 25~26번 섹션의 사람 라벨과 대조한 뒤에 정한다.)

| 지표 | 의미 | 해석 주의 |
|---|---|---|
| `words_ratio_to_nc` | 추출 단어 수 / NewsCatcher `word_count` | NewsCatcher 값도 추정치다. 1에서 멀수록 확인 대상일 뿐, 틀렸다는 뜻이 아니다. |
| `title_similarity` | NewsCatcher 제목과 추출 제목의 문자열 유사도(0~1) | 매체가 제목에 사이트명을 붙이면 낮아진다. |
| `hormuz_mentions` | 본문에서 "hormuz" 등장 횟수 | 0이어도 관련 기사일 수 있다(맥락 기사). 점수 0 기사 검토에 쓴다. |
| `boilerplate_hit_types` | 쿠키/구독/로그인/JS 요구/오류/저작권/공유 문구의 흔적 종류 | 단순 키워드 검출이라 오탐이 있다. 사람 검토 우선순위용이다. |
| `repeated_line_ratio` | 같은 줄이 반복되는 비율 | 줄이 2개 미만이면 계산하지 않는다. |
| `date_diff_days` | 추출 날짜 - NewsCatcher `published_date` (일) | 기준 날짜는 계속 NewsCatcher 값이다. 추출 날짜로 덮어쓰지 않는다. |
| 추출기 간 일치도 | 단어 집합 Jaccard / 포함도(containment) | 같은 기사의 본문을 서로 다르게 잘랐는지 보는 용도다. |

In [8]:
import difflib

BOILERPLATE_PATTERNS = {
    "cookie": r"\bcookies?\b|cookie policy|accept all|manage (?:your )?preferences",
    "subscribe": r"subscribe (?:now|today|to)|sign up for (?:our|the)|newsletter",
    "login_paywall": r"sign in to (?:read|continue)|log in to (?:read|continue)|create (?:a )?free account|already a subscriber|premium content|unlock (?:this )?article",
    "js_required": r"enable javascript|javascript is (?:disabled|required)|please enable cookies",
    "error_page": r"page not found|404 error|access denied|something went wrong|temporarily unavailable",
    "rights": r"all rights reserved|copyright \d{4}",
    "share_nav": r"share on (?:facebook|twitter)|follow us on|related articles|most read",
}
_BOILERPLATE_RE = {k: re.compile(v, re.I) for k, v in BOILERPLATE_PATTERNS.items()}


def boilerplate_hits(text):
    return [name for name, rx in _BOILERPLATE_RE.items() if rx.search(text)]


def repeated_line_ratio(text):
    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]
    if len(lines) < 2:
        return np.nan
    return 1 - len(set(lines)) / len(lines)


def token_set(text):
    return set(re.findall(r"\w+", text.lower()))


br = benchmark_results.copy()
br["body_nonempty"] = br["body"].notna()
br["success"] = br["extraction_status"].eq("success")

words = pd.to_numeric(br["word_count_extracted"], errors="coerce")
nc_words = pd.to_numeric(br["nc_word_count"], errors="coerce")
br["words_ratio_to_nc"] = words / nc_words

title_lookup = bench_run.set_index("article_id")["title"].to_dict()
br["title_similarity"] = [
    difflib.SequenceMatcher(None, normalize_title(title_lookup.get(a)), normalize_title(t)).ratio()
    if isinstance(t, str) and t.strip() and normalize_title(title_lookup.get(a)) else np.nan
    for a, t in zip(br["article_id"], br["extracted_title"])
]

br["hormuz_mentions"] = br["body"].apply(lambda t: t.lower().count("hormuz") if isinstance(t, str) else np.nan)
_hits = br["body"].apply(lambda t: boilerplate_hits(t) if isinstance(t, str) else None)
br["boilerplate_hit_types"] = _hits.apply(lambda h: ",".join(h) if isinstance(h, list) else None)
br["boilerplate_hit_count"] = _hits.apply(lambda h: len(h) if isinstance(h, list) else np.nan)
br["repeated_line_ratio"] = br["body"].apply(lambda t: repeated_line_ratio(t) if isinstance(t, str) else np.nan)

_pub = pd.to_datetime(br["article_id"].map(bench_run.set_index("article_id")["published_date"]), errors="coerce", utc=True)
_ext = pd.to_datetime(br["extracted_date"], errors="coerce", utc=True)
br["date_diff_days"] = (_ext - _pub).dt.total_seconds() / 86400

# ---- 추출기 간 일치도 (본문이 있는 추출기끼리만) ----
agreement_rows = []
for article_id, grp in br[br["body_nonempty"]].groupby("article_id"):
    sets = {r.extractor_used: token_set(r.body) for r in grp.itertuples()}
    names = sorted(sets)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = sets[names[i]], sets[names[j]]
            if not a or not b:
                continue
            inter = len(a & b)
            agreement_rows.append({
                "article_id": article_id, "extractor_a": names[i], "extractor_b": names[j],
                "jaccard": inter / len(a | b), "containment": inter / min(len(a), len(b)),
            })
pairwise_agreement = pd.DataFrame(agreement_rows, columns=["article_id", "extractor_a", "extractor_b", "jaccard", "containment"])
benchmark_diagnostics = br

print("진단 지표 계산 완료:", len(benchmark_diagnostics), "행 /", len(pairwise_agreement), "개 추출기 쌍")

진단 지표 계산 완료: 60 행 / 26 개 추출기 쌍


## 24. 요약 표

> **중요:** `success`는 "본문이 비어 있지 않다"는 뜻이다. 성공률이 높다고 좋은 추출기가 아니다. 이 표는 어디에서 실패가 몰리는지(도메인 유형, 국가, 월, 관련성 점수, 페이월 표시)를 보기 위한 것이다.

- **주 표본(`stratified_main`)**: 표본 가중치를 적용한 성공률과 가중치 없는 성공률을 함께 보여준다. 가중치를 적용한 값이 코퍼스 전체에 대한 추정에 가깝다. (표본이 작으면 불확실성이 크다.)
- **보충 표본(`fundus_supplement`)**: Fundus 지원 매체에서 세 추출기를 비교하는 용도로만 본다. 코퍼스 전체 비율로 해석하지 않는다.

In [9]:
def success_rate_table(df, by=None, weighted=False):
    d = df.copy()
    d["_ok"] = d["success"].astype(float)
    if weighted:
        d["_w"] = d["sample_weight"].astype(float)
        d["_okw"] = d["_ok"] * d["_w"]
    group_cols = ["extractor_used"] + ([by] if by else [])
    grouped = d.groupby(group_cols, dropna=False)
    out = grouped.agg(n=("_ok", "size"), success_rate=("_ok", "mean"))
    if weighted:
        out["success_rate_weighted"] = grouped["_okw"].sum() / grouped["_w"].sum()
    return out.reset_index()


main_diag = benchmark_diagnostics[benchmark_diagnostics["sample_origin"] == "stratified_main"]
supp_diag = benchmark_diagnostics[benchmark_diagnostics["sample_origin"] == "fundus_supplement"]

summary_tables = {}

print("=" * 70)
print("1) 가져오기 상태 (기사 단위)")
print("=" * 70)
_fetch_view = fetch_log_df.merge(bench_run[["article_id", "sample_origin"]], on="article_id", how="left")
summary_tables["fetch_status"] = pd.crosstab(_fetch_view["sample_origin"], _fetch_view["fetch_status"], margins=True)
display(summary_tables["fetch_status"])

print("\n" + "=" * 70)
print("2) 추출기별 상태 (행 = 추출기, 열 = 상태)")
print("=" * 70)
for origin, d in [("stratified_main", main_diag), ("fundus_supplement", supp_diag)]:
    if len(d):
        print(f"\n[{origin}]")
        summary_tables[f"status_{origin}"] = pd.crosstab(d["extractor_used"], d["extraction_status"], margins=True)
        display(summary_tables[f"status_{origin}"])

print("\n" + "=" * 70)
print("3) 주 표본 성공률 (가중치 없음 vs 가중치 적용)")
print("=" * 70)
if len(main_diag):
    summary_tables["success_main"] = success_rate_table(main_diag, weighted=True)
    display(summary_tables["success_main"])

print("\n" + "=" * 70)
print("4) 주 표본: 층별 성공률 (n이 작은 칸은 해석하지 않는다)")
print("=" * 70)
for col in ["flag_aggregator_review", "hormuz_relevance_score", "fundus_supported", "nc_paid_content", "month"]:
    if len(main_diag) and col in main_diag.columns:
        tbl = success_rate_table(main_diag, by=col)
        piv = tbl.pivot(index=col, columns="extractor_used", values="success_rate").round(3)
        piv["n_articles"] = main_diag.groupby(col)["article_id"].nunique()
        summary_tables[f"success_by_{col}"] = piv
        print(f"\n[{col}]")
        display(piv)

if len(main_diag):
    _top_countries = main_diag["nc_country"].fillna("(unknown)").value_counts().head(8).index
    _tmp = main_diag.assign(nc_country=main_diag["nc_country"].fillna("(unknown)"))
    _tmp = _tmp[_tmp["nc_country"].isin(_top_countries)]
    tbl = success_rate_table(_tmp, by="nc_country")
    piv = tbl.pivot(index="nc_country", columns="extractor_used", values="success_rate").round(3)
    piv["n_articles"] = _tmp.groupby("nc_country")["article_id"].nunique()
    summary_tables["success_by_country_top"] = piv
    print("\n[nc_country 상위 8개]")
    display(piv)

print("\n" + "=" * 70)
print("5) Fundus 지원 도메인 기사만 비교 (세 추출기 모두 실행 가능한 기사)")
print("=" * 70)
_sup = benchmark_diagnostics[benchmark_diagnostics["fundus_supported"].astype(str).str.lower().eq("true")]
if len(_sup):
    summary_tables["success_fundus_supported"] = success_rate_table(_sup, by="sample_origin")
    display(summary_tables["success_fundus_supported"])
else:
    print("Fundus 지원 도메인 기사가 표본에 없습니다.")

print("\n" + "=" * 70)
print("6) 본문 길이 / NewsCatcher 단어 수 대비 비율 (본문이 있는 행만)")
print("=" * 70)
_has = benchmark_diagnostics[benchmark_diagnostics["body_nonempty"]]
if len(_has):
    summary_tables["length_quantiles"] = (
        _has.groupby("extractor_used")[["body_length", "word_count_extracted", "words_ratio_to_nc"]]
        .quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(2).unstack()
    )
    display(summary_tables["length_quantiles"])

print("\n" + "=" * 70)
print("7) 추출기 간 일치도")
print("=" * 70)
if len(pairwise_agreement):
    summary_tables["agreement"] = (
        pairwise_agreement.groupby(["extractor_a", "extractor_b"])[["jaccard", "containment"]]
        .describe().round(3)
    )
    display(summary_tables["agreement"][[("jaccard", "count"), ("jaccard", "50%"), ("containment", "50%")]])

print("\n" + "=" * 70)
print("8) 가져오기는 성공했지만 세 추출기 모두 본문이 없는 기사 (직접 HTML 확인 대상)")
print("=" * 70)
_ok_ids = set(fetch_log_df.loc[fetch_log_df["fetch_status"] == "ok", "article_id"])
_nonempty_by_article = benchmark_diagnostics.groupby("article_id")["body_nonempty"].any()
all_empty_ids = [a for a in _ok_ids if not _nonempty_by_article.get(a, False)]
print(f"{len(all_empty_ids)}건 / 가져오기 성공 {len(_ok_ids)}건")
display(bench_run[bench_run["article_id"].isin(all_empty_ids)][["article_id", "domain", "url", "sample_origin"]].head(20))

print("\n[주의] 위 성공률은 '본문이 비어 있지 않음'만 뜻한다. 본문 품질은 사람 라벨(섹션 25~26)로 확인한다.")
if BENCH_RUN_LIMIT:
    print(f"[주의] BENCH_RUN_LIMIT={BENCH_RUN_LIMIT} 시험 실행 결과이므로 수치를 해석하지 마세요.")

1) 가져오기 상태 (기사 단위)


fetch_status,blocked,dead_link,ok,All
sample_origin,,,,
fundus_supplement,0,0,5,5
stratified_main,2,1,12,15
All,2,1,17,20



2) 추출기별 상태 (행 = 추출기, 열 = 상태)

[stratified_main]


extraction_status,blocked,dead_link,extractor_error,success,unsupported_publisher,All
extractor_used,,,,,,
fundus,2,1,0,1,11,15
news-please,2,1,1,11,0,15
trafilatura,2,1,0,12,0,15
All,6,3,1,24,11,45



[fundus_supplement]


...


7) 추출기 간 일치도


jaccard        containment
                          count    50%         50%
extractor_a extractor_b                           
fundus      news-please     5.0  0.948         1.0
            trafilatura     5.0  0.985         1.0
news-please trafilatura    16.0  0.973         1.0


8) 가져오기는 성공했지만 세 추출기 모두 본문이 없는 기사 (직접 HTML 확인 대상)
0건 / 가져오기 성공 17건


,article_id,domain,url,sample_origin



[주의] 위 성공률은 '본문이 비어 있지 않음'만 뜻한다. 본문 품질은 사람 라벨(섹션 25~26)로 확인한다.
[주의] BENCH_RUN_LIMIT=20 시험 실행 결과이므로 수치를 해석하지 마세요.


## 25. 사람이 라벨링할 시트 만들기

자동 지표만으로는 "이 본문이 진짜 기사 본문인가"를 알 수 없으므로, 일부 기사를 사람이 직접 확인한다. 시트는 **기사 한 건당 추출기별로 한 줄**이다. 같은 기사를 세 추출기가 어떻게 뽑았는지 나란히 볼 수 있고, 추출기 순서는 기사마다 무작위로 섞는다. (특정 추출기를 먼저 본다는 편향을 줄이기 위함)

- `LABEL_ARTICLES_MAIN`: 주 표본에서 라벨링할 기사 수. `LABEL_ARTICLES_FUNDUS`: Fundus 보충 표본에서 라벨링할 기사 수.
- `LABEL_ALL_EMPTY_N`: 가져오기는 성공했지만 세 추출기가 모두 비어 있는 기사. **추출기가 놓친 기사(재현율 문제)** 를 확인하기 위해 일부를 포함한다. `html_path`의 파일이나 `url`을 열어 실제 페이지에 본문이 있는지 확인한다.
- 이 규모(기본 약 130건)는 시험적(pilot) 라벨링이다. 비율의 95% 오차를 ±5%p 이내로 줄이려면 약 385건이 필요하므로, 결과가 애매하면 숫자를 늘려 다시 만든다.

### 라벨 컬럼 (비어 있는 칸을 채운다. 모두 `1` / `0`)

| 컬럼 | 1 = | 0 = |
|---|---|---|
| `label_is_article_body` | 기사 본문이다 | 메뉴/오류/쿠키/목록 등이다 |
| `label_complete` | 본문이 처음부터 끝까지 충분히 들어 있다 | 일부만 있거나 심하게 잘렸다 |
| `label_contaminated` | 메뉴·광고·관련기사 등 본문 외 텍스트가 섞였다 | 깨끗하다 |
| `label_title_ok` | 추출 제목이 기사와 맞다 | 틀렸거나 사이트명뿐이다 |
| `label_language_en` | 본문이 영어다 | 아니다 |
| `label_syndicated_dup` | 다른 기사의 재게시/복제로 보인다 | 아니다 |
| `label_paywall_or_blocked` | 페이월/차단 문구가 보인다 | 아니다 |
| `label_page_has_article_body` | (모두 비어 있는 기사 전용) 실제 페이지에 기사 본문이 있다 | 없다 |
| `label_notes` | 자유 메모 | |

라벨링이 끝나면 **`06_benchmark_label_sheet_labeled.csv`** 라는 이름으로(CSV UTF-8) 저장한다. Excel에서 저장할 때는 "CSV UTF-8(쉼표로 분리)"를 선택한다.

In [10]:
LABEL_ARTICLES_MAIN = 100
LABEL_ARTICLES_FUNDUS = 30
LABEL_ALL_EMPTY_N = 15
LABEL_SEED = _seed

LABEL_COLUMNS = [
    "label_is_article_body", "label_complete", "label_contaminated", "label_title_ok",
    "label_language_en", "label_syndicated_dup", "label_paywall_or_blocked",
    "label_page_has_article_body", "label_notes",
]


def clip_text(text, head=800, tail=400):
    if not isinstance(text, str):
        return None
    flat = text.replace("\n", " ⏎ ")
    return flat if len(flat) <= head + tail else flat[:head] + "  […]  " + flat[-tail:]


rng = np.random.default_rng(LABEL_SEED)
_meta = bench_run.set_index("article_id")

_has_body_ids = benchmark_diagnostics.loc[benchmark_diagnostics["body_nonempty"], "article_id"].unique()
_main_ids = [a for a in _has_body_ids if _meta.loc[a, "sample_origin"] == "stratified_main"]
_supp_ids = [a for a in _has_body_ids if _meta.loc[a, "sample_origin"] == "fundus_supplement"]

pick_main = list(rng.choice(_main_ids, size=min(LABEL_ARTICLES_MAIN, len(_main_ids)), replace=False)) if _main_ids else []
pick_supp = list(rng.choice(_supp_ids, size=min(LABEL_ARTICLES_FUNDUS, len(_supp_ids)), replace=False)) if _supp_ids else []
pick_empty = list(rng.choice(all_empty_ids, size=min(LABEL_ALL_EMPTY_N, len(all_empty_ids)), replace=False)) if all_empty_ids else []

selected = pick_main + pick_supp + pick_empty
order_of_article = {a: i for i, a in enumerate(rng.permutation(selected))}

rows = []
for article_id in selected:
    meta = _meta.loc[article_id]
    base = {
        "article_id": article_id, "sample_origin": meta["sample_origin"], "domain": meta["domain"],
        "fundus_supported": meta["fundus_supported"], "url": meta["url"], "nc_title": meta["title"],
        "nc_word_count": meta["nc_word_count"],
    }
    grp = benchmark_diagnostics[(benchmark_diagnostics["article_id"] == article_id) & benchmark_diagnostics["body_nonempty"]]
    if article_id in pick_empty:
        fetch_row = fetch_log_df[fetch_log_df["article_id"] == article_id].iloc[0]
        rows.append({**base, "extractor_used": "(all_empty)", "extracted_title": None, "word_count_extracted": 0,
                     "body_head_800_tail_400": "(세 추출기 모두 본문 없음 - 원본 HTML/URL을 열어 확인)",
                     "html_path": fetch_row.get("html_path"), "all_extractors_empty": True})
        continue
    for r in grp.itertuples():
        rows.append({**base, "extractor_used": r.extractor_used, "extracted_title": r.extracted_title,
                     "word_count_extracted": r.word_count_extracted,
                     "body_head_800_tail_400": clip_text(r.body), "html_path": r.html_path,
                     "all_extractors_empty": False})

label_sheet = pd.DataFrame(rows)
if len(label_sheet):
    label_sheet["_article_order"] = label_sheet["article_id"].map(order_of_article)
    label_sheet["_rand"] = rng.random(len(label_sheet))
    label_sheet = (label_sheet.sort_values(["_article_order", "_rand"]).drop(columns=["_article_order", "_rand"]).reset_index(drop=True))
    label_sheet.insert(0, "label_row_id", range(1, len(label_sheet) + 1))
    for col in LABEL_COLUMNS:
        label_sheet[col] = ""

print("=" * 70)
print("LABEL SHEET")
print("=" * 70)
print(f"라벨링 대상 기사: 주 표본 {len(pick_main)} + Fundus 보충 {len(pick_supp)} + 전부 비어 있음 {len(pick_empty)} = {len(selected)}건")
print(f"시트 행 수: {len(label_sheet):,}")
if len(label_sheet):
    display(label_sheet[["label_row_id", "article_id", "domain", "extractor_used", "word_count_extracted"]].head(12))

LABEL SHEET
라벨링 대상 기사: 주 표본 12 + Fundus 보충 5 + 전부 비어 있음 0 = 17건
시트 행 수: 38


,label_row_id,article_id,domain,extractor_used,word_count_extracted
0,1,d8faf841276334fb,www.euronews.com,fundus,1380.0
1,2,d8faf841276334fb,www.euronews.com,trafilatura,1380.0
2,3,d8faf841276334fb,www.euronews.com,news-please,1347.0
3,4,58b65969e1dd7566,www.lemonde.fr,news-please,373.0
4,5,58b65969e1dd7566,www.lemonde.fr,trafilatura,388.0
5,6,58b65969e1dd7566,www.lemonde.fr,fundus,415.0
6,7,001f7ed882f1e4db,www.jordannews.jo,trafilatura,259.0
7,8,001f7ed882f1e4db,www.jordannews.jo,news-please,263.0
8,9,c6cb59fe3f05bbf6,www.indexbox.io,trafilatura,1774.0
9,10,c6cb59fe3f05bbf6,www.indexbox.io,news-please,485.0


## 26. 라벨링 결과 평가 (라벨 파일이 있을 때만 실행됨)

`06_benchmark_label_sheet_labeled.csv`가 있으면 추출기별로 다음을 계산한다.

- 각 라벨 항목의 비율과 **Wilson 95% 신뢰구간** (표본이 작으면 구간이 넓다는 점을 그대로 보여준다)
- "쓸 만한 본문"(`is_article_body = 1` 그리고 `complete = 1` 그리고 `contaminated = 0`)의 비율
- 같은 기사에서 두 추출기를 비교하는 **짝지은(paired) 비교** — 한쪽만 쓸 만한 기사 수와 McNemar 정확검정 p값
- Fundus 지원 도메인 기사만 따로 본 비교
- 가져오기는 성공했지만 세 추출기가 모두 비었던 기사 중 실제 페이지에 본문이 있던 비율

**기본 추출기와 fallback은 이 표를 보고 사람이 정한다.** 이 셀은 자동으로 선택하지 않는다.

In [ ]:
from math import comb, sqrt


def wilson_interval(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z ** 2 / n
    center = (p + z ** 2 / (2 * n)) / denom
    half = z * sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / denom
    return (max(0.0, center - half), min(1.0, center + half))


def mcnemar_exact_p(b, c):
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / (2 ** n))


def read_labeled_csv(path):
    for enc in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(path, encoding=enc, dtype={"article_id": str})
        except UnicodeDecodeError:
            continue
    raise ValueError("라벨 파일의 인코딩을 읽을 수 없습니다. CSV UTF-8로 다시 저장하세요.")


label_summary = None
if not BENCH_LABELED_CSV.exists():
    print("라벨링된 파일이 아직 없습니다.")
    print(f"1) {BENCH_LABEL_SHEET_CSV.name} 을 Excel로 열어 label_* 칸을 채운다 (1/0).")
    print(f"2) 'CSV UTF-8'로 {BENCH_LABELED_CSV.name} 이라는 이름으로 같은 폴더에 저장한다.")
    print("3) 이 셀을 다시 실행한다.")
else:
    lab = read_labeled_csv(BENCH_LABELED_CSV)
    for col in LABEL_COLUMNS[:-1]:
        lab[col] = pd.to_numeric(lab[col], errors="coerce")

    body_rows = lab[lab["extractor_used"] != "(all_empty)"].copy()
    body_rows["label_usable"] = np.where(
        body_rows[["label_is_article_body", "label_complete", "label_contaminated"]].isna().any(axis=1), np.nan,
        ((body_rows["label_is_article_body"] == 1) & (body_rows["label_complete"] == 1) & (body_rows["label_contaminated"] == 0)).astype(float),
    )

    metric_cols = ["label_is_article_body", "label_complete", "label_contaminated", "label_title_ok",
                   "label_language_en", "label_paywall_or_blocked", "label_usable"]
    rows = []
    for extractor, grp in body_rows.groupby("extractor_used"):
        for col in metric_cols:
            valid = grp[col].dropna()
            k, n = int(valid.sum()), len(valid)
            lo, hi = wilson_interval(k, n)
            rows.append({"extractor": extractor, "metric": col, "k": k, "n_labeled": n,
                         "rate": (k / n) if n else np.nan, "ci95_low": lo, "ci95_high": hi})
    label_summary = pd.DataFrame(rows)
    print("추출기별 라벨 비율 (Wilson 95% CI)")
    display(label_summary.pivot(index="metric", columns="extractor", values="rate").round(3))
    display(label_summary[label_summary["metric"] == "label_usable"].round(3))

    print("\n같은 기사 paired 비교 (label_usable 기준)")
    wide = body_rows.pivot_table(index="article_id", columns="extractor_used", values="label_usable", aggfunc="first")
    paired_rows = []
    cols = list(wide.columns)
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            sub = wide[[cols[i], cols[j]]].dropna()
            b = int(((sub[cols[i]] == 1) & (sub[cols[j]] == 0)).sum())
            c = int(((sub[cols[i]] == 0) & (sub[cols[j]] == 1)).sum())
            paired_rows.append({"extractor_a": cols[i], "extractor_b": cols[j], "n_both_labeled": len(sub),
                                "only_a_usable": b, "only_b_usable": c, "mcnemar_exact_p": round(mcnemar_exact_p(b, c), 4)})
    display(pd.DataFrame(paired_rows))

    sup_rows = body_rows[body_rows["fundus_supported"].astype(str).str.lower().eq("true")]
    if len(sup_rows):
        print("\nFundus 지원 도메인 기사만 (label_usable 비율)")
        display(sup_rows.groupby("extractor_used")["label_usable"].agg(["count", "mean"]).round(3))

    empties = lab[lab["extractor_used"] == "(all_empty)"]
    if len(empties):
        valid = pd.to_numeric(empties["label_page_has_article_body"], errors="coerce").dropna()
        print(f"\n세 추출기 모두 비었던 기사 중 실제 페이지에 본문이 있던 비율: {int(valid.sum())}/{len(valid)}")

    print("\n[결정은 사람이 한다] 위 표를 보고 아래 값을 직접 채운다. (이 셀은 자동으로 선택하지 않는다)")

# 라벨 결과를 확인한 뒤 직접 채운다. 다음 단계(전체 본문 추출)의 입력이 된다.
PRIMARY_EXTRACTOR = None     # 예: "trafilatura"
FALLBACK_ORDER = []          # 예: ["news-please"]

## 27. 벤치마크 산출물 저장

생성 파일 (모두 `hormuz_2026_data/` 안, 기존 `01_*`, `05_*` 파일은 덮어쓰지 않음)

| 파일 | 내용 |
|---|---|
| `06_benchmark_fetch_log.jsonl` | 가져오기 시도 기록(실시간 누적, 재개용) |
| `06_benchmark_fetch_log.csv` | 위 기록의 표 형태 |
| `06_benchmark_extraction_results.csv` | 추출기 × 기사 결과 + 진단 지표 (본문 포함) |
| `06_benchmark_pairwise_agreement.csv` | 추출기 간 일치도 |
| `06_benchmark_summary_*.csv` | 섹션 24 요약 표 |
| `06_benchmark_label_sheet.csv` | 사람 라벨링용 시트 |
| `benchmark_html_cache/` | 원본 HTML (Git에 올리지 않는다) |

In [ ]:
fetch_log_df.to_csv(BENCH_FETCH_LOG_CSV, index=False, encoding="utf-8-sig")
benchmark_diagnostics.to_csv(BENCH_RESULTS_CSV, index=False, encoding="utf-8-sig")
pairwise_agreement.to_csv(BENCH_AGREEMENT_CSV, index=False, encoding="utf-8-sig")
label_sheet.to_csv(BENCH_LABEL_SHEET_CSV, index=False, encoding="utf-8-sig")

saved = [BENCH_FETCH_LOG_JSONL, BENCH_FETCH_LOG_CSV, BENCH_RESULTS_CSV, BENCH_AGREEMENT_CSV, BENCH_LABEL_SHEET_CSV]
for name, table in summary_tables.items():
    path = DATA_DIR / f"06_benchmark_summary_{name}.csv"
    table.to_csv(path, encoding="utf-8-sig")
    saved.append(path)

print("저장 완료")
for p in saved:
    print("-", p)
print("- (HTML 캐시)", BENCH_HTML_DIR)
if BENCH_RUN_LIMIT:
    print(f"\n[주의] BENCH_RUN_LIMIT={BENCH_RUN_LIMIT} 시험 실행 결과가 저장되었습니다. 전체 실행 시 같은 이름으로 갱신됩니다.")

# 벤치마크 단계 완료 기준

이 섹션을 끝까지 실행하면 다음 상태가 된다.

**벤치마크 표본**  
→ **한 번만 가져와 HTML 저장 (접속 실패/차단/페이월 의심 기록)**  
→ **같은 HTML에 Fundus / news-please / trafilatura 적용**  
→ **자동 진단 지표 + 층별 요약 (가중치 적용 성공률 포함)**  
→ **사람 라벨링 시트 생성**  
→ **(라벨링 후) 추출기별 정확도와 신뢰구간, 짝지은 비교**

## 다음 단계 (이 노트북에서는 아직 하지 않음)

1. 라벨링 결과를 보고 `PRIMARY_EXTRACTOR`와 `FALLBACK_ORDER`를 사람이 결정한다.
2. 결정된 조합으로 `extraction_candidates` 전체(427,875건)에 본문 추출을 적용한다. 이때도 HTML을 먼저 저장하고, 실패한 기사를 지우지 않고 `extraction_status`와 `failure_reason`으로 남긴다.
3. 추출 성공률을 도메인 유형·국가·월·관련성 점수·`nc_paid_content`별로 보고해, 최종 코퍼스가 "추출에 성공한 기사들"로 좁아지는 편향을 확인한다.
4. 본문 품질 필터(길이, boilerplate, 언어, 오류 페이지)는 **라벨링으로 정당화한 기준**으로 적용한다.
5. 본문 유사도(MinHash/SimHash)로 중복/신디케이션 그룹을 만든다. (제목이 같다는 이유로 삭제하지 않는다.)

## 이 단계에서도 하지 않는 것

- 본문 추출 전에 메타데이터만으로 본문 품질을 판단해 기사를 삭제
- paywall / Cloudflare / robots 제한 우회
- 추출기가 준 날짜로 연구 기준 `published_date` 덮어쓰기
- 벤치마크 성공률만으로 기본 추출기 결정